In [ ]:
print('Initializingh the project')

In [ ]:
# Install the CUDA-enabled versions of torch and torchvision specifically for this session
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# 1. Kaggle Setup

In [ ]:
# Please change these two variables in the bottom of the file `../.env `  to your Kaggle username and key, which you can find in your Kaggle account settings. 
# This will allow you to download the dataset directly from Kaggle using the Kaggle API.

# KAGGLE_USERNAME = 'your_kaggle_username'
# KAGGLE_KEY = 'your_kaggle_key'



# 2. Data Download using kaggle

In [ ]:
%run ..\assignment3\dataset.py


# 3. Cleaning Data and producing these files 

- ../data/processed/counsel_chat_clean.csv   (RAG corpus)
- ../data/processed/crisis_val.csv           (~15% — validation)
- ../data/processed/crisis_train.csv         (~70% — classifier training)
- ../data/processed/crisis_test.csv          (~15% — held-out test)

In [ ]:
# ============================================================================
# Data Cleaning & Preparation — Mental Health Chatbot
#
# Paste this entire block into a single Jupyter notebook cell and run it.
# Prerequisite: run `python dataset.py` first to download raw data.
#
# Produces:
#   ../data/processed/counsel_chat_clean.csv   (RAG corpus)
#   ../data/processed/crisis_train.csv         (~70% — classifier training)
#   ../data/processed/crisis_val.csv           (~15% — validation)
#   ../data/processed/crisis_test.csv          (~15% — held-out test)
# ============================================================================

import re
from pathlib import Path

import pandas as pd
from datasets import load_dataset
from sklearn.model_selection import train_test_split

# --- Locate project root -----------------------------------------------------
# Walks upward from the notebook's working directory to find the folder that
# contains the `data/` directory (created by dataset.py).
_cwd = Path.cwd().resolve()
PROJ_ROOT = next(
    (p for p in [_cwd, *_cwd.parents] if (p / "data").exists()),
    _cwd,
)
RAW_DATA_DIR       = PROJ_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJ_ROOT / "data" / "processed"
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
print(f"📁 PROJ_ROOT: {PROJ_ROOT}")

# --- Paths & constants -------------------------------------------------------
CRISIS_RAW_PATH    = RAW_DATA_DIR / "crisis_raw.csv"
COUNSEL_RAW_PATH   = RAW_DATA_DIR / "counsel_chat_raw.csv"
COUNSEL_CLEAN_PATH = PROCESSED_DATA_DIR / "counsel_chat_clean.csv"
CRISIS_TRAIN_PATH  = PROCESSED_DATA_DIR / "crisis_train.csv"
CRISIS_VAL_PATH    = PROCESSED_DATA_DIR / "crisis_val.csv"
CRISIS_TEST_PATH   = PROCESSED_DATA_DIR / "crisis_test.csv"

RANDOM_SEED    = 42
MIN_ANSWER_LEN = 50    # Counsel Chat: drop very short therapist answers
MIN_TEXT_LEN   = 20    # Crisis: drop very short posts


# --- Helpers -----------------------------------------------------------------
def clean_text(text):
    """Normalize whitespace, strip URLs, drop non-ASCII characters."""
    if not isinstance(text, str):
        return ""
    text = re.sub(r"http\S+", " ", text)          # strip URLs
    text = re.sub(r"[^\x00-\x7F]+", " ", text)    # ASCII only
    text = re.sub(r"\s+", " ", text)              # collapse whitespace
    return text.strip()


# --- 1. Counsel Chat → RAG corpus --------------------------------------------
print("\n📚 Processing Counsel Chat ...")
counsel_df = load_dataset("nbertagnolli/counsel-chat", split="train").to_pandas()
counsel_df.to_csv(COUNSEL_RAW_PATH, index=False)

counsel_df["questionText"] = counsel_df["questionText"].apply(clean_text)
counsel_df["answerText"]   = counsel_df["answerText"].apply(clean_text)
counsel_df = counsel_df.dropna(subset=["questionText", "answerText"])
counsel_df = counsel_df[counsel_df["answerText"].str.len() >= MIN_ANSWER_LEN]

counsel_df["document"] = (
    "Question: " + counsel_df["questionText"]
    + "\nAnswer: " + counsel_df["answerText"]
)
counsel_df[["questionText", "answerText", "topic", "document"]].to_csv(
    COUNSEL_CLEAN_PATH, index=False
)
print(f"✅ Counsel Chat: {len(counsel_df):,} rows -> {COUNSEL_CLEAN_PATH.name}")


# --- 2. Crisis dataset → train/val/test --------------------------------------
print("\n🩺 Processing Crisis dataset ...")
if not CRISIS_RAW_PATH.exists():
    raise FileNotFoundError(
        f"crisis_raw.csv not found at {CRISIS_RAW_PATH}. "
        "Run `python dataset.py` first to download it from Kaggle."
    )

crisis_df = pd.read_csv(CRISIS_RAW_PATH)
if "Unnamed: 0" in crisis_df.columns:
    crisis_df = crisis_df.drop(columns=["Unnamed: 0"])

crisis_df = crisis_df[["text", "class"]].copy()
crisis_df.columns = ["text", "label"]
crisis_df["text"] = crisis_df["text"].apply(clean_text)
crisis_df = crisis_df[crisis_df["text"].str.len() >= MIN_TEXT_LEN].dropna()

crisis_df["label_binary"] = crisis_df["label"].map({"suicide": 1, "non-suicide": 0})
crisis_df = crisis_df.dropna(subset=["label_binary"])
crisis_df["label_binary"] = crisis_df["label_binary"].astype(int)

train, temp = train_test_split(
    crisis_df,
    test_size=0.30,
    random_state=RANDOM_SEED,
    stratify=crisis_df["label_binary"],
)
val, test = train_test_split(
    temp,
    test_size=0.50,
    random_state=RANDOM_SEED,
    stratify=temp["label_binary"],
)

train.to_csv(CRISIS_TRAIN_PATH, index=False)
val.to_csv(CRISIS_VAL_PATH,   index=False)
test.to_csv(CRISIS_TEST_PATH, index=False)

print(
    f"✅ Crisis splits | Train: {len(train):,} | "
    f"Val: {len(val):,} | Test: {len(test):,}"
)
print("\n🎉 All processed datasets ready.")

# 4.1 Training the Model ( Optional : The already trained model is downloadable by runing Cell 4.2 instead )

In [ ]:
# # ============================================================================
# # Crisis Classifier Training — DistilBERT fine-tuning
# #
# # Paste this entire block into a single Jupyter notebook cell and run it.
# # Prerequisite: data/processed/crisis_train.csv and crisis_val.csv exist
# #               (run the cleaning cell or `python clean.py` first).
# #
# # Saves the final model to: <PROJ_ROOT>/models/crisis_classifier/
# # ============================================================================

# from pathlib import Path

# import numpy as np
# import pandas as pd
# import torch
# from datasets import Dataset
# from sklearn.metrics import (
#     accuracy_score,
#     f1_score,
#     precision_score,
#     recall_score,
# )
# from transformers import (
#     AutoModelForSequenceClassification,
#     AutoTokenizer,
#     Trainer,
#     TrainingArguments,
# )

# # --- Locate project root -----------------------------------------------------
# _cwd = Path.cwd().resolve()
# PROJ_ROOT = next(
#     (p for p in [_cwd, *_cwd.parents]
#      if (p / "data").exists() or (p / ".env").exists()),
#     _cwd,
# )
# TRAIN_PATH = PROJ_ROOT / "data" / "processed" / "crisis_train.csv"
# VAL_PATH   = PROJ_ROOT / "data" / "processed" / "crisis_val.csv"
# MODEL_DIR  = PROJ_ROOT / "models" / "crisis_classifier"

# # --- Hyperparameters ---------------------------------------------------------
# MODEL_NAME  = "distilbert-base-uncased"
# MAX_LENGTH  = 128
# SEED        = 42

# SAMPLE      = False       # True  = quick smoke run (10k rows)
#                          # False = full dataset
# SAMPLE_SIZE = 10_000
# VAL_SAMPLE  = 2_000

# NUM_EPOCHS       = 3
# TRAIN_BATCH_SIZE = 16
# EVAL_BATCH_SIZE  = 32

# # --- Device ------------------------------------------------------------------
# if torch.cuda.is_available():
#     DEVICE = "cuda"
#     print("✅ Using CUDA GPU")
# elif torch.backends.mps.is_available():
#     DEVICE = "mps"
#     print("✅ Using Apple Silicon MPS")
# else:
#     DEVICE = "cpu"
#     print("⚠️  Using CPU — full training will be very slow. Consider Colab.")

# print(f"📁 PROJ_ROOT: {PROJ_ROOT}")
# print(f"📂 Output:    {MODEL_DIR}")

# # --- Load & sample data ------------------------------------------------------
# if not TRAIN_PATH.exists() or not VAL_PATH.exists():
#     raise FileNotFoundError(
#         f"Crisis splits not found in {TRAIN_PATH.parent}. "
#         "Run the cleaning cell (or clean.py) first."
#     )

# train_df = pd.read_csv(TRAIN_PATH)[["text", "label_binary"]].dropna()
# val_df   = pd.read_csv(VAL_PATH)[["text", "label_binary"]].dropna()

# if SAMPLE:
#     train_df = train_df.sample(n=min(SAMPLE_SIZE, len(train_df)), random_state=SEED)
#     val_df   = val_df.sample(  n=min(VAL_SAMPLE,  len(val_df)),   random_state=SEED)
#     print(f"⚠️  SAMPLE MODE — Train: {len(train_df):,} | Val: {len(val_df):,}")
# else:
#     print(f"✅ FULL MODE — Train: {len(train_df):,} | Val: {len(val_df):,}")

# # HF Trainer expects the label column to be called 'labels'
# train_df = train_df.rename(columns={"label_binary": "labels"})
# val_df   = val_df.rename(columns={"label_binary": "labels"})


# # --- Tokenize ----------------------------------------------------------------
# print("\n🔤 Loading tokenizer ...")
# tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# def _tok_batch(batch):
#     return tokenizer(
#         batch["text"],
#         padding="max_length",
#         truncation=True,
#         max_length=MAX_LENGTH,
#     )

# print("🔤 Tokenising ...")
# train_ds = Dataset.from_pandas(train_df, preserve_index=False).map(_tok_batch, batched=True)
# val_ds   = Dataset.from_pandas(val_df,   preserve_index=False).map(_tok_batch, batched=True)


# # --- Model -------------------------------------------------------------------
# print(f"\n🤖 Loading base model '{MODEL_NAME}' ...")
# model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)


# # --- Metrics -----------------------------------------------------------------
# def compute_metrics(eval_pred):
#     logits, labels = eval_pred
#     preds = np.argmax(logits, axis=1)
#     return {
#         "accuracy":  accuracy_score(labels, preds),
#         "f1":        f1_score(labels, preds),
#         "precision": precision_score(labels, preds),
#         "recall":    recall_score(labels, preds),
#     }


# # --- Training ----------------------------------------------------------------
# MODEL_DIR.mkdir(parents=True, exist_ok=True)

# args = TrainingArguments(
#     output_dir=str(MODEL_DIR),
#     num_train_epochs=NUM_EPOCHS,
#     per_device_train_batch_size=TRAIN_BATCH_SIZE,
#     per_device_eval_batch_size=EVAL_BATCH_SIZE,
#     eval_strategy="epoch",
#     save_strategy="epoch",
#     load_best_model_at_end=True,
#     metric_for_best_model="f1",
#     logging_steps=50,
#     seed=SEED,
#     report_to="none",   # disable wandb / tensorboard auto-init
# )

# trainer = Trainer(
#     model=model,
#     args=args,
#     train_dataset=train_ds,
#     eval_dataset=val_ds,
#     compute_metrics=compute_metrics,
# )

# print("\n🏋️  Training started ...")
# trainer.train()

# print(f"\n💾 Saving final model to {MODEL_DIR} ...")
# trainer.save_model(str(MODEL_DIR))
# tokenizer.save_pretrained(str(MODEL_DIR))
# print("✅ Training complete.")

# 4.2 Downloading the previously trained model for the project 

In [ ]:
# --- Device ------------------------------------------------------------------
import torch
if torch.cuda.is_available():
    DEVICE = "cuda"
    print("✅ Using CUDA GPU")
elif torch.backends.mps.is_available():
    DEVICE = "mps"
    print("✅ Using Apple Silicon MPS")
else:
    DEVICE = "cpu"
    print("⚠️  Using CPU — full training will be very slow. Consider Colab.")


In [ ]:
print('Starting the model download')
# ============================================================================
# Download fine-tuned Crisis Classifier model from Google Drive
#
# Paste this entire block into a single Jupyter notebook cell and run it.
# Requires: pip install gdown
#
# Produces:
#   ../models/crisis_classifier/
#       ├── config.json
#       ├── model.safetensors
#       ├── tokenizer.json
#       ├── tokenizer_config.json
#       └── training_args.bin
# ============================================================================

from pathlib import Path

import gdown

# --- Locate project root -----------------------------------------------------
# Walks upward from the notebook's working directory until it finds a folder
# that has either `data/` or `.env` in it (i.e. the project root).
_cwd = Path.cwd().resolve()
PROJ_ROOT = next(
    (p for p in [_cwd, *_cwd.parents]
     if (p / "data").exists() or (p / ".env").exists()),
    _cwd,
)
CRISIS_MODEL_DIR = PROJ_ROOT / "models" / "crisis_classifier"
CRISIS_MODEL_DIR.mkdir(parents=True, exist_ok=True)
print(f"📁 Model dir: {CRISIS_MODEL_DIR}")

# --- Google Drive file IDs ---------------------------------------------------
FILES = {
    "config.json":           "16tvn1X7CgSvMNWRQxpdHNqyBOfH_YwXD",
    "model.safetensors":     "1kFfjErvjm2EfJW5u2DMYxXwWoDnFmQaG",
    "tokenizer.json":        "1IzTCTPK7ElMYSSj5Vl5x57M1LTJ3h3VB",
    "tokenizer_config.json": "1IHtzJajxLyd8sEJXE7_mENPUC-LZHj-j",
    "training_args.bin":     "1qHRNifMtwQs5ebq_mDAj3xkIuiof08kF",
}

# --- Download ----------------------------------------------------------------
downloaded, skipped = 0, 0
for filename, file_id in FILES.items():
    out_path = CRISIS_MODEL_DIR / filename
    if out_path.exists():
        print(f"⏭️  {filename} already exists, skipping.")
        skipped += 1
        continue
    print(f"⬇️  Downloading {filename} ...")
    gdown.download(id=file_id, output=str(out_path), quiet=False)
    downloaded += 1

print(
    f"\n✅ Crisis classifier ready | Downloaded: {downloaded} | "
    f"Skipped: {skipped} | Location: {CRISIS_MODEL_DIR}"
)

# 5. Test The predictions 

In [ ]:
print('hello')

In [ ]:
# ============================================================================
# Crisis Classifier — Inference Smoke Test
#
# Paste this entire block into a single Jupyter notebook cell and run it.
# Prerequisite: model files in <PROJ_ROOT>/models/crisis_classifier/
#               (run download_model.py / its notebook cell first).
# ============================================================================

from pathlib import Path

import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# --- Locate project root -----------------------------------------------------
# Walks upward from the notebook's working directory until it finds a folder
# that has either `data/` or `.env` in it.
_cwd = Path.cwd().resolve()
PROJ_ROOT = next(
    (p for p in [_cwd, *_cwd.parents]
     if (p / "data").exists() or (p / ".env").exists()),
    _cwd,
)
CRISIS_MODEL_DIR = PROJ_ROOT / "models" / "crisis_classifier"
MAX_LENGTH       = 128
THRESHOLD        = 0.7
DEVICE           = "cuda" if torch.cuda.is_available() else "cpu"

print(f"📁 Model dir: {CRISIS_MODEL_DIR}")
print(f"🖥️  Device:    {DEVICE}")

# --- Load model --------------------------------------------------------------
if not CRISIS_MODEL_DIR.exists():
    raise FileNotFoundError(
        f"Model not found at {CRISIS_MODEL_DIR}. "
        "Run the download_model notebook cell first to fetch it from Google Drive."
    )

# AutoTokenizer / AutoModel pick the right class automatically.
# (The model dir ships tokenizer.json — fast tokenizer — but no vocab.txt,
#  so the slow DistilBertTokenizer would fail to load.)
tokenizer = AutoTokenizer.from_pretrained(str(CRISIS_MODEL_DIR))
model     = AutoModelForSequenceClassification.from_pretrained(str(CRISIS_MODEL_DIR))
model.to(DEVICE).eval()
print("✅ Model loaded\n")


# --- Inference ---------------------------------------------------------------
def is_crisis(text, threshold=THRESHOLD):
    """Score a single message for crisis content."""
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.softmax(logits, dim=1)
    crisis_prob = probs[0][1].item()

    return {
        "is_crisis":  crisis_prob >= threshold,
        "confidence": round(crisis_prob, 4),
        "label":      "crisis" if crisis_prob >= threshold else "non-crisis",
        "method":     "model",
    }


# --- Test it -----------------------------------------------------------------
test_messages = [
    "I have been feeling really anxious lately and don't know what to do 😃",
    "I want to kill myself, I can't take this anymore 😃",
    "Can you help me with some breathing exercises?",
    "I've been having dark thoughts and feel like ending it all",
    "I'm struggling with my relationship and feeling lost",
]

print("Testing messages:")
print("-" * 60)
for msg in test_messages:
    result = is_crisis(msg)
    flag = "🚨" if result["is_crisis"] else "✅"
    print(
        f"{flag} [{result['label'].upper()}] "
        f"confidence: {result['confidence']:.2%}"
    )
    print(f"   {msg[:80]}")
    print()

# 6. Evaluate the Predictions

In [ ]:
# ============================================================================
# Crisis Classifier Evaluation — held-out test set
#
# Paste this entire block into a single Jupyter notebook cell and run it.
# Prerequisites:
#   - Trained model in <PROJ_ROOT>/models/crisis_classifier/
#   - <PROJ_ROOT>/data/processed/crisis_test.csv
#
# Outputs:
#   - reports/crisis_classifier_results.csv
#   - reports/figures/confusion_matrix.png
# ============================================================================

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import torch
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# --- Locate project root -----------------------------------------------------
_cwd = Path.cwd().resolve()
PROJ_ROOT = next(
    (p for p in [_cwd, *_cwd.parents]
     if (p / "data").exists() or (p / ".env").exists()),
    _cwd,
)
MODEL_DIR    = PROJ_ROOT / "models" / "crisis_classifier"
TEST_PATH    = PROJ_ROOT / "data" / "processed" / "crisis_test.csv"
REPORTS_DIR  = PROJ_ROOT / "reports"
FIGURES_DIR  = REPORTS_DIR / "figures"
CM_FIG_PATH  = FIGURES_DIR / "confusion_matrix.png"
RESULTS_PATH = REPORTS_DIR / "crisis_classifier_results.csv"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

MAX_LENGTH = 128
BATCH_SIZE = 32
THRESHOLD  = 0.7
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"

print(f"📁 PROJ_ROOT: {PROJ_ROOT}")
print(f"🖥️  Device:    {DEVICE}")


# --- Dataset class -----------------------------------------------------------
class CrisisDataset(Dataset):
    def __init__(self, texts, labels, tokenizer):
        self.encodings = tokenizer(
            list(texts),
            truncation=True,
            padding="max_length",
            max_length=MAX_LENGTH,
            return_tensors="pt",
        )
        self.labels = list(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids":      self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx],
            "label":          self.labels[idx],
        }


# --- Load test data ----------------------------------------------------------
if not TEST_PATH.exists():
    raise FileNotFoundError(
        f"Test split not found at {TEST_PATH}. "
        "Run the cleaning cell (or clean.py) first."
    )

print("\n📊 Loading test data ...")
df = pd.read_csv(TEST_PATH)[["text", "label_binary"]].dropna()
print(f"✅ Test set: {len(df):,} rows")


# --- Load model --------------------------------------------------------------
if not MODEL_DIR.exists():
    raise FileNotFoundError(
        f"Model not found at {MODEL_DIR}. "
        "Run the download_model cell or train cell first."
    )

print(f"\n🤖 Loading model from {MODEL_DIR} ...")
tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR))
model     = AutoModelForSequenceClassification.from_pretrained(str(MODEL_DIR))
model.to(DEVICE).eval()


# --- Tokenize & build loader -------------------------------------------------
print("🔤 Tokenising ...")
dataset = CrisisDataset(
    df["text"].tolist(),
    df["label_binary"].tolist(),
    tokenizer,
)
loader = DataLoader(dataset, batch_size=BATCH_SIZE)


# --- Predict -----------------------------------------------------------------
print("\n🔮 Running predictions ...")
all_preds, all_labels = [], []

with torch.no_grad():
    for i, batch in enumerate(loader):
        input_ids      = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)

        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
        ).logits
        probs = torch.softmax(logits, dim=1)
        preds = (probs[:, 1] >= THRESHOLD).int().cpu().tolist()

        all_preds.extend(preds)
        all_labels.extend(batch["label"].tolist())

        if i % 10 == 0:
            print(f"  Batch {i}/{len(loader)} ...")


# --- Metrics -----------------------------------------------------------------
acc  = accuracy_score(all_labels, all_preds)
f1   = f1_score(all_labels, all_preds)
prec = precision_score(all_labels, all_preds)
rec  = recall_score(all_labels, all_preds)
cm   = confusion_matrix(all_labels, all_preds)

print("\n" + "=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)
print(f"Accuracy:  {acc:.4f} ({acc * 100:.2f}%)")
print(f"F1 Score:  {f1:.4f} ({f1 * 100:.2f}%)")
print(f"Precision: {prec:.4f} ({prec * 100:.2f}%)")
print(f"Recall:    {rec:.4f} ({rec * 100:.2f}%)")
print("\nClassification Report:")
print(classification_report(
    all_labels, all_preds,
    target_names=["Non-Crisis", "Crisis"],
))


# --- Confusion matrix plot ---------------------------------------------------
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=["Non-Crisis", "Crisis"],
    yticklabels=["Non-Crisis", "Crisis"],
    ax=ax,
)
ax.set_title("Confusion Matrix — Crisis Classifier", fontsize=13)
ax.set_ylabel("Actual")
ax.set_xlabel("Predicted")
plt.tight_layout()
plt.savefig(CM_FIG_PATH, dpi=150)
plt.show()
print(f"\n✅ Confusion matrix saved to {CM_FIG_PATH}")


# --- Save metrics CSV --------------------------------------------------------
results = pd.DataFrame({
    "Metric": ["Accuracy", "F1 Score", "Precision", "Recall"],
    "Score":  [acc, f1, prec, rec],
})
results.to_csv(RESULTS_PATH, index=False)
print(f"✅ Results saved to {RESULTS_PATH}")

# 7. Buildig RAG

## 7.1 Step 1 Build the RAG retrieval system

In [ ]:
# ============================================================================
# Cell 1 (v2) — Build RAG Vector Index, grouped by unique question
#
# Same as v1 but groups counsel_chat_clean.csv by questionText so each unique
# question becomes one record. All therapist answers for that question are
# concatenated into metadata['answerText'], separated by ANSWER_SEPARATOR.
#
# This fixes the duplicate-question issue we saw in retrieval (top-5 was
# pulling the same question 5x with different answers, instead of 5 different
# relevant questions).
#
# Set FORCE_REBUILD = True to wipe and re-index. Default True here since v1
# already exists and we want to replace it.
# ============================================================================

from pathlib import Path

import chromadb
import pandas as pd
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer

# --- Locate project root -----------------------------------------------------
_cwd = Path.cwd().resolve()
PROJ_ROOT = next(
    (p for p in [_cwd, *_cwd.parents]
     if (p / "data").exists() or (p / ".env").exists()),
    _cwd,
)
COUNSEL_CLEAN_PATH = PROJ_ROOT / "data" / "processed" / "counsel_chat_clean.csv"
CHROMA_DIR         = PROJ_ROOT / "models" / "chroma_db"
CHROMA_DIR.mkdir(parents=True, exist_ok=True)

EMBED_MODEL_NAME = "all-MiniLM-L6-v2"
COLLECTION_NAME  = "counsel_chat"
BATCH_SIZE       = 64
ANSWER_SEPARATOR = "\n\n---\n\n"
FORCE_REBUILD    = True   # ← True: wipe v1 and rebuild grouped index

print(f"📁 PROJ_ROOT:    {PROJ_ROOT}")
print(f"📂 Chroma dir:   {CHROMA_DIR}")
print(f"🧠 Embed model:  {EMBED_MODEL_NAME}")


# --- Load + group source data -----------------------------------------------
if not COUNSEL_CLEAN_PATH.exists():
    raise FileNotFoundError(
        f"{COUNSEL_CLEAN_PATH} not found. Run the cleaning cell first."
    )

df = pd.read_csv(COUNSEL_CLEAN_PATH)
df = df.dropna(subset=["questionText", "answerText"]).reset_index(drop=True)
print(f"\n📊 Loaded {len(df):,} raw Q&A rows")

# Group by unique question. Aggregate all answers (joined with separator),
# pick the most common topic for that question (in case answers disagree),
# and count how many answers there are.
grouped = (
    df.groupby("questionText", as_index=False)
      .agg(
          answers=("answerText", lambda s: ANSWER_SEPARATOR.join(s.astype(str))),
          n_answers=("answerText", "count"),
          topic=("topic", lambda s: s.mode().iloc[0] if not s.mode().empty else "unknown"),
      )
)
print(f"📊 After grouping: {len(grouped):,} unique questions "
      f"(avg {grouped['n_answers'].mean():.1f} answers each)")


# --- Init Chroma client ------------------------------------------------------
client = chromadb.PersistentClient(
    path=str(CHROMA_DIR),
    settings=Settings(anonymized_telemetry=False),
)

existing = [c.name for c in client.list_collections()]

if COLLECTION_NAME in existing and not FORCE_REBUILD:
    coll = client.get_collection(COLLECTION_NAME)
    print(
        f"\n⏭️  Collection '{COLLECTION_NAME}' already exists with "
        f"{coll.count():,} items — skipping."
    )
else:
    if COLLECTION_NAME in existing:
        print(f"\n🗑️  Deleting existing '{COLLECTION_NAME}' collection ...")
        client.delete_collection(COLLECTION_NAME)

    coll = client.create_collection(
        name=COLLECTION_NAME,
        metadata={"hnsw:space": "cosine"},
    )

    # --- Embed ---------------------------------------------------------------
    print(f"\n🧠 Loading embedding model '{EMBED_MODEL_NAME}' ...")
    embed_model = SentenceTransformer(EMBED_MODEL_NAME)

    questions = grouped["questionText"].tolist()
    print(f"🔢 Embedding {len(questions):,} unique questions ...")
    embeddings = embed_model.encode(
        questions,
        batch_size=BATCH_SIZE,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

    # --- Build metadata payload ---------------------------------------------
    ids = [f"counsel_{i}" for i in range(len(grouped))]
    documents = questions
    metadatas = [
        {
            "source":       "counsel_chat",
            "topic":        str(row["topic"]),
            "questionText": str(row["questionText"]),
            "answerText":   str(row["answers"]),       # all answers, separator-joined
            "n_answers":    int(row["n_answers"]),
        }
        for _, row in grouped.iterrows()
    ]

    # --- Insert in batches --------------------------------------------------
    print(f"💾 Inserting into Chroma in batches of {BATCH_SIZE} ...")
    for start in range(0, len(ids), BATCH_SIZE):
        end = start + BATCH_SIZE
        coll.add(
            ids=ids[start:end],
            documents=documents[start:end],
            embeddings=embeddings[start:end].tolist(),
            metadatas=metadatas[start:end],
        )

    print(f"\n✅ Indexed {coll.count():,} unique questions into '{COLLECTION_NAME}'.")
    print(f"   Persisted to {CHROMA_DIR}")

In [ ]:
# ============================================================================
# Cell 2 — Sanity-check RAG Retrieval
#
# Loads the persisted Chroma DB, embeds a handful of representative user
# messages, and prints the top-k retrieved counsel chat questions + topics.
#
# The point: eyeball whether retrieval is pulling sensible matches before
# we wire it into the LLM. If retrieval is bad, generation will be bad.
# ============================================================================

from pathlib import Path

import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer

# --- Locate project root -----------------------------------------------------
_cwd = Path.cwd().resolve()
PROJ_ROOT = next(
    (p for p in [_cwd, *_cwd.parents]
     if (p / "data").exists() or (p / ".env").exists()),
    _cwd,
)
CHROMA_DIR       = PROJ_ROOT / "models" / "chroma_db"
EMBED_MODEL_NAME = "all-MiniLM-L6-v2"
COLLECTION_NAME  = "counsel_chat"
TOP_K            = 5


# --- Load Chroma + embedding model ------------------------------------------
client = chromadb.PersistentClient(
    path=str(CHROMA_DIR),
    settings=Settings(anonymized_telemetry=False),
)
coll = client.get_collection(COLLECTION_NAME)
print(f"📊 Collection '{COLLECTION_NAME}' has {coll.count():,} items\n")

embed_model = SentenceTransformer(EMBED_MODEL_NAME)


# --- Helper ------------------------------------------------------------------
def retrieve(query: str, k: int = TOP_K):
    """Embed the query and return top-k matches with metadata + similarity."""
    q_emb = embed_model.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True,
    )[0].tolist()

    res = coll.query(
        query_embeddings=[q_emb],
        n_results=k,
    )
    # Chroma returns nested lists (one per query). We pass one query, so [0].
    out = []
    for doc, meta, dist in zip(
        res["documents"][0],
        res["metadatas"][0],
        res["distances"][0],
    ):
        # cosine distance -> similarity. Both in [0, 2]; sim in [-1, 1].
        similarity = 1.0 - dist
        out.append({
            "similarity":   round(similarity, 3),
            "topic":        meta.get("topic", "?"),
            "questionText": doc,
            "answerText":   meta.get("answerText", ""),
        })
    return out


# --- Test queries (deliberately varied) -------------------------------------
test_queries = [
    "I've been feeling really anxious lately and don't know what to do",
    "How do I deal with grief after losing a parent?",
    "My partner and I keep fighting, I think we should break up",
    "I can't sleep at night because my mind keeps racing",
    "I feel disconnected from everyone, like nobody understands me",
    "How do I help my teenage son who's depressed?",
]

for q in test_queries:
    print("=" * 80)
    print(f"🔍 Query: {q}")
    print("=" * 80)
    results = retrieve(q, k=TOP_K)
    for i, r in enumerate(results, 1):
        print(f"\n  [{i}] sim={r['similarity']:.3f}  topic={r['topic']}")
        print(f"      Q: {r['questionText'][:160]}")
        print(f"      A: {r['answerText'][:160]} ...")
    print()

## 7.2 Step 2 Curated knowledge base (Team B's "structured knowledge")

In [ ]:
# ============================================================================
# Cell 1 — Build the Curated Knowledge Base JSON
#
# Writes ~30 entries covering:
#   - CBT techniques        (~12 entries)
#   - Mindfulness/grounding (~8 entries)
#   - Psychoeducation       (~10 entries)
#
# Output:
#   data/external/knowledge_base.json
#
# Schema per entry:
#   { id, category, title, tags, description, how_to,
#     when_useful, source, disclaimer_level }
#
# IMPORTANT: This is LLM-drafted content, NOT pulled from a verified clinical
# source database. Before deploying, sanity-check the entries — especially
# psychoeducation ones. Suggested cross-refs: NIMH, Beyond Blue, Black Dog
# Institute, the actual Beck/Hayes/Linehan source texts cited below.
# ============================================================================

import json
from pathlib import Path

# --- Locate project root -----------------------------------------------------
_cwd = Path.cwd().resolve()
PROJ_ROOT = next(
    (p for p in [_cwd, *_cwd.parents]
     if (p / "data").exists() or (p / ".env").exists()),
    _cwd,
)
KB_PATH = PROJ_ROOT / "data" / "external" / "knowledge_base.json"
KB_PATH.parent.mkdir(parents=True, exist_ok=True)
print(f"📁 Output: {KB_PATH}")


# ============================================================================
# CBT TECHNIQUES
# ============================================================================
cbt_entries = [
    {
        "id": "cbt_001",
        "category": "CBT",
        "title": "Cognitive Reframing",
        "tags": ["anxiety", "depression", "negative thoughts", "rumination", "cognitive distortion"],
        "description": "Cognitive reframing is the practice of noticing an automatic negative thought, examining the evidence for and against it, and considering an alternative, more balanced perspective. The goal is not forced positivity — it is accuracy. Many distressing thoughts are exaggerated or one-sided, and gently testing them often reveals more workable interpretations.",
        "how_to": "1) Notice the thought when you feel a strong emotional shift. 2) Write it down word-for-word. 3) Ask: what is the evidence this is true? What is the evidence against it? 4) Ask: what would I tell a close friend who had this thought? 5) Write a more balanced alternative thought that accounts for both sides. 6) Notice how you feel after considering the alternative.",
        "when_useful": "Useful when you notice 'always/never' thinking, mind-reading, catastrophizing, or harsh self-criticism.",
        "source": "Beck, J. (2011). Cognitive Behavior Therapy: Basics and Beyond (2nd ed.). Guilford Press.",
        "disclaimer_level": "low",
    },
    {
        "id": "cbt_002",
        "category": "CBT",
        "title": "Thought Record",
        "tags": ["anxiety", "depression", "negative thoughts", "self-monitoring", "journaling"],
        "description": "A thought record is a structured way to slow down and examine an upsetting situation. Writing down the situation, the feelings, the automatic thought, the evidence, and the rebalanced thought makes patterns visible over time. Many people find that simply writing the thought down reduces its grip.",
        "how_to": "Use five columns: Situation (what happened), Feeling (1-10 intensity), Automatic Thought, Evidence For/Against, Balanced Thought. Fill it in shortly after a difficult moment, not in the heat of it. Review your records weekly to spot recurring themes.",
        "when_useful": "Helpful when emotions feel overwhelming or confusing, or when you want to track patterns across days.",
        "source": "Greenberger, D. & Padesky, C. (2015). Mind Over Mood (2nd ed.). Guilford Press.",
        "disclaimer_level": "low",
    },
    {
        "id": "cbt_003",
        "category": "CBT",
        "title": "Behavioral Activation",
        "tags": ["depression", "low motivation", "withdrawal", "isolation", "fatigue"],
        "description": "Behavioral activation rests on a simple observation: when we feel low, we do less, and doing less makes us feel lower. Reversing the cycle by scheduling small, achievable activities — even when motivation is absent — often lifts mood before motivation returns. The principle is action first, motivation later.",
        "how_to": "1) List activities that used to bring you pleasure or a sense of accomplishment. 2) Rate each from 1-10 for how doable it feels right now. 3) Pick one or two from the easier end. 4) Schedule them at specific times this week. 5) Do them whether or not you feel like it. 6) Note how you felt before and after.",
        "when_useful": "Especially useful when depression makes everything feel pointless or exhausting.",
        "source": "Martell, C., Dimidjian, S., & Herman-Dunn, R. (2010). Behavioral Activation for Depression. Guilford Press.",
        "disclaimer_level": "low",
    },
    {
        "id": "cbt_004",
        "category": "CBT",
        "title": "ABC Model",
        "tags": ["anxiety", "anger", "emotional regulation", "cognitive distortion"],
        "description": "The ABC model separates an Activating event from the Belief about it and the Consequent emotion. The insight is that the same event can produce very different emotions depending on how it is interpreted. By identifying the belief, you create space to consider whether it serves you.",
        "how_to": "When upset, write down: A — what actually happened, B — what you told yourself about it, C — how you felt and what you did. Then ask whether the belief was an interpretation rather than a fact, and what alternative belief might fit the same evidence.",
        "when_useful": "Useful for strong reactions that feel disproportionate to the situation.",
        "source": "Ellis, A. (1962). Reason and Emotion in Psychotherapy. Stuart.",
        "disclaimer_level": "low",
    },
    {
        "id": "cbt_005",
        "category": "CBT",
        "title": "Decatastrophizing",
        "tags": ["anxiety", "worry", "panic", "catastrophizing"],
        "description": "Decatastrophizing is the practice of asking 'what if the worst-case actually happened — could I cope?' instead of 'how do I prevent this from ever happening?' For most worries, the catastrophic outcome is unlikely, and even if it occurred, you would have resources to handle it. Naming this can shrink the worry.",
        "how_to": "1) Identify the worry: 'I'm afraid that ___ will happen.' 2) Ask: realistically, how likely is this? 3) If it did happen, what is the worst that would actually follow? 4) What would I do to cope? 5) Has anything like this happened before, and how did I get through it?",
        "when_useful": "Helpful when worry spirals into worst-case scenarios that feel overwhelming.",
        "source": "Clark, D. & Beck, A. (2011). Cognitive Therapy of Anxiety Disorders. Guilford Press.",
        "disclaimer_level": "low",
    },
    {
        "id": "cbt_006",
        "category": "CBT",
        "title": "Common Cognitive Distortions",
        "tags": ["negative thoughts", "cognitive distortion", "self-criticism", "anxiety", "depression"],
        "description": "Cognitive distortions are recurring patterns of biased thinking. Recognizing them by name reduces their power. Common ones include all-or-nothing thinking, mind-reading, catastrophizing, personalizing, 'should' statements, emotional reasoning, and disqualifying the positive. Most people have a handful of habitual distortions.",
        "how_to": "Read through a list of distortions and circle two or three that feel familiar. For one week, label your distressing thoughts with the distortion name as they happen. Naming creates distance — 'that's catastrophizing' is easier to challenge than the raw thought.",
        "when_useful": "Foundational skill that pairs with thought records and reframing.",
        "source": "Burns, D. (1980). Feeling Good: The New Mood Therapy. Avon.",
        "disclaimer_level": "low",
    },
    {
        "id": "cbt_007",
        "category": "CBT",
        "title": "Problem-Solving Therapy",
        "tags": ["stress", "decision-making", "overwhelm", "depression"],
        "description": "Problem-solving therapy turns vague distress into concrete steps. When problems feel impossible, it is often because they are tangled together. Breaking them into definable pieces and brainstorming options — without judging the options yet — restores a sense of agency.",
        "how_to": "1) Write the problem in one specific sentence. 2) List as many possible solutions as you can, including bad ones, without filtering. 3) For each, note pros and cons. 4) Pick one to try, even if imperfect. 5) Plan the first small step and when you'll do it. 6) After trying, evaluate and adjust.",
        "when_useful": "When you feel stuck, overwhelmed, or paralyzed by a life situation.",
        "source": "Nezu, A., Nezu, C., & D'Zurilla, T. (2013). Problem-Solving Therapy: A Treatment Manual. Springer.",
        "disclaimer_level": "low",
    },
    {
        "id": "cbt_008",
        "category": "CBT",
        "title": "Exposure Hierarchy",
        "tags": ["anxiety", "phobia", "avoidance", "panic", "social anxiety"],
        "description": "Exposure works on the principle that anxiety naturally decreases when you stay in a feared situation long enough, rather than escaping it. Building a hierarchy lets you start small and work up. Each exposure that you complete teaches the brain that the feared outcome did not occur.",
        "how_to": "1) List situations you avoid related to your fear. 2) Rate each 0-100 for anticipated anxiety. 3) Order them from easiest to hardest. 4) Start with an item rated 30-40. 5) Stay in the situation until your anxiety drops at least halfway. 6) Repeat until that step feels manageable, then move up.",
        "when_useful": "Best practiced with a therapist for moderate-to-severe phobias, panic, or PTSD. Self-directed exposure can help with milder avoidance.",
        "source": "Foa, E., Hembree, E., & Rothbaum, B. (2007). Prolonged Exposure Therapy for PTSD. Oxford University Press.",
        "disclaimer_level": "medium",
    },
    {
        "id": "cbt_009",
        "category": "CBT",
        "title": "Self-Compassion Practice",
        "tags": ["self-criticism", "shame", "depression", "perfectionism"],
        "description": "Self-compassion involves treating yourself with the kindness you would extend to a friend. It includes three elements: self-kindness instead of self-judgement, common humanity (recognizing that suffering is universal), and mindful acceptance of difficult emotions. Research suggests it is more sustainable than self-esteem because it does not depend on success.",
        "how_to": "When you notice harsh self-talk, pause and ask: what would I say to a friend in this exact situation? Place a hand on your chest, take a slow breath, and say silently: 'this is a moment of suffering; suffering is part of life; may I be kind to myself.'",
        "when_useful": "Helpful for chronic self-criticism, perfectionism, or shame.",
        "source": "Neff, K. & Germer, C. (2018). The Mindful Self-Compassion Workbook. Guilford Press.",
        "disclaimer_level": "low",
    },
    {
        "id": "cbt_010",
        "category": "CBT",
        "title": "Worry Time",
        "tags": ["anxiety", "worry", "rumination", "generalized anxiety"],
        "description": "Worry time paradoxically reduces worry by giving it a designated slot. Instead of trying to suppress worry (which usually backfires), you postpone worries to a specific 15-30 minute window. Most worries lose urgency by the time the slot arrives.",
        "how_to": "1) Pick a daily 15-30 minute window — not just before bed. 2) When a worry arises outside that window, jot it down briefly and tell yourself you'll attend to it later. 3) During worry time, review the list and worry deliberately. 4) Notice how many worries no longer feel pressing.",
        "when_useful": "Helpful for chronic worriers and people with generalized anxiety.",
        "source": "Borkovec, T. & Inz, J. (1990). The nature of worry in generalized anxiety disorder. Behaviour Research and Therapy, 28(2), 153-158.",
        "disclaimer_level": "low",
    },
    {
        "id": "cbt_011",
        "category": "CBT",
        "title": "Sleep Hygiene Basics",
        "tags": ["insomnia", "sleep", "anxiety", "depression"],
        "description": "Sleep hygiene refers to behaviors that protect sleep quality. While none of these alone cure insomnia, together they build a context where sleep can occur naturally. The most evidence-based ones involve consistent timing and bedroom-as-sleep-cue.",
        "how_to": "Keep wake-up time consistent, even on weekends. Avoid caffeine after midday. Reserve the bed for sleep — not work, scrolling, or worrying. If you cannot sleep within ~20 minutes, get up, do something quiet in dim light, and return when sleepy. Limit screens for the hour before bed.",
        "when_useful": "First-line for sleep difficulties before considering medication.",
        "source": "Morin, C. & Espie, C. (2003). Insomnia: A Clinical Guide to Assessment and Treatment. Springer.",
        "disclaimer_level": "low",
    },
    {
        "id": "cbt_012",
        "category": "CBT",
        "title": "Activity Scheduling",
        "tags": ["depression", "low motivation", "fatigue", "structure"],
        "description": "Activity scheduling adds structure to days that feel shapeless. Depression often blurs days into one long fog; planning specific small activities at specific times restores rhythm. The act of completing planned activities — even tiny ones — builds momentum.",
        "how_to": "On Sunday, sketch a rough plan for the week with one activity per morning, afternoon, and evening. Mix obligations with small pleasures (a coffee out, a walk, a call). Keep activities small enough that you'd feel slightly silly skipping them. Tick them off as you complete them.",
        "when_useful": "Pairs naturally with behavioral activation for depression.",
        "source": "Lewinsohn, P., Sullivan, J., & Grosscup, S. (1980). Changing reinforcing events: An approach to the treatment of depression. Psychotherapy, 17(3), 322-334.",
        "disclaimer_level": "low",
    },
]


# ============================================================================
# MINDFULNESS / GROUNDING
# ============================================================================
mindfulness_entries = [
    {
        "id": "mind_001",
        "category": "Mindfulness",
        "title": "Box Breathing",
        "tags": ["anxiety", "panic", "stress", "breathing", "grounding"],
        "description": "Box breathing slows the breath to four equal counts, which activates the parasympathetic nervous system and reduces acute physiological arousal. It is widely used because it is portable, takes under two minutes, and works in almost any setting.",
        "how_to": "Inhale through the nose for a count of 4. Hold for 4. Exhale through the mouth for 4. Hold empty for 4. Repeat for 4-6 cycles. If 4 feels too long, start with 3.",
        "when_useful": "Acute anxiety, before a stressful event, panic onset, or to settle before sleep.",
        "source": "Adapted from US Navy SEAL tactical breathing protocols and broader pranayama traditions.",
        "disclaimer_level": "low",
    },
    {
        "id": "mind_002",
        "category": "Mindfulness",
        "title": "5-4-3-2-1 Grounding",
        "tags": ["anxiety", "panic", "dissociation", "grounding", "trauma", "flashback"],
        "description": "5-4-3-2-1 is a sensory grounding exercise that interrupts anxious or dissociative loops by anchoring attention in the present body and environment. By cycling through five senses, you give the mind a concrete task that competes with rumination.",
        "how_to": "Look around and silently name: 5 things you can see, 4 things you can feel (your feet on the floor, fabric on your skin), 3 things you can hear, 2 things you can smell, 1 thing you can taste. Take your time with each. Repeat if needed.",
        "when_useful": "Panic attacks, dissociation, flashbacks, or any moment when you feel disconnected from the present.",
        "source": "Najavits, L. (2002). Seeking Safety: A Treatment Manual for PTSD and Substance Abuse. Guilford Press.",
        "disclaimer_level": "low",
    },
    {
        "id": "mind_003",
        "category": "Mindfulness",
        "title": "Body Scan",
        "tags": ["stress", "tension", "sleep", "mindfulness", "relaxation"],
        "description": "A body scan involves slowly directing attention through different regions of the body, noticing sensations without trying to change them. It builds the skill of observing the body without immediately reacting, and often releases held tension as a side effect.",
        "how_to": "Lie down or sit comfortably. Starting at the toes, bring attention to each region of the body in turn — feet, calves, knees, thighs, hips, abdomen, chest, hands, arms, shoulders, neck, face. Spend 20-30 seconds at each, simply noticing whatever is there: warmth, tension, pulsing, or nothing. No need to fix anything.",
        "when_useful": "Stress, tension, falling asleep, or as a daily mindfulness practice.",
        "source": "Kabat-Zinn, J. (1990). Full Catastrophe Living. Bantam.",
        "disclaimer_level": "low",
    },
    {
        "id": "mind_004",
        "category": "Mindfulness",
        "title": "Progressive Muscle Relaxation",
        "tags": ["anxiety", "tension", "stress", "sleep", "relaxation"],
        "description": "PMR teaches the contrast between tension and relaxation by deliberately tensing and then releasing muscle groups. Many people hold chronic tension without noticing; the explicit release shows the nervous system a target state of relaxation.",
        "how_to": "Working from feet to head, tense each muscle group for 5 seconds, then release for 10-15 seconds, noticing the difference. Order: feet, calves, thighs, glutes, abdomen, hands (clench fists), arms, shoulders (raise to ears), face (squint and tighten). One full cycle takes about 10 minutes.",
        "when_useful": "Generalized anxiety, tension headaches, falling asleep.",
        "source": "Jacobson, E. (1938). Progressive Relaxation. University of Chicago Press.",
        "disclaimer_level": "low",
    },
    {
        "id": "mind_005",
        "category": "Mindfulness",
        "title": "Urge Surfing",
        "tags": ["addiction", "cravings", "self-harm urges", "impulse control", "DBT"],
        "description": "Urge surfing treats an urge as a wave that rises, peaks, and falls. Instead of fighting or giving in, you observe the sensation in the body — where you feel it, how intense it is, how it changes — until the wave passes. Most urges peak in 20-30 minutes if you don't act on them.",
        "how_to": "When the urge arises, locate it physically: where in the body? What does it feel like — heat, pressure, restlessness? Rate intensity 0-10. Breathe slowly and observe the sensation as if you were a curious witness. Notice how it shifts. Don't fight, don't act, just watch the wave.",
        "when_useful": "Cravings, urges to self-harm, impulses to lash out — moments where 'doing nothing' is the goal.",
        "source": "Marlatt, G. & Donovan, D. (2005). Relapse Prevention. Guilford Press.",
        "disclaimer_level": "medium",
    },
    {
        "id": "mind_006",
        "category": "Mindfulness",
        "title": "Mindful Walking",
        "tags": ["anxiety", "depression", "rumination", "mindfulness", "movement"],
        "description": "Mindful walking combines gentle movement with present-moment attention. Unlike sitting meditation, it gives the restless mind something to do and works well for people who find seated practice difficult. Even 10 minutes can shift mood.",
        "how_to": "Walk at a slightly slower pace than usual, ideally outdoors. Pay attention to the sensation of each foot landing and lifting. Notice the air on your skin, sounds around you, what you see. When the mind wanders to plans or worries, gently return attention to walking.",
        "when_useful": "Rumination, mild depression, sensory overwhelm, or as a daily reset.",
        "source": "Kabat-Zinn, J. (1994). Wherever You Go, There You Are. Hyperion.",
        "disclaimer_level": "low",
    },
    {
        "id": "mind_007",
        "category": "Mindfulness",
        "title": "STOP Skill",
        "tags": ["stress", "emotional regulation", "DBT", "impulse control"],
        "description": "STOP is a four-step micro-practice for moments when you feel about to react in a way you might regret. It buys a small but crucial pause between trigger and response. With practice, the pause becomes automatic.",
        "how_to": "S — Stop. Don't move, don't speak. T — Take a breath, slow and full. O — Observe: what's happening in my body, in my mind, around me? P — Proceed mindfully: choose a response rather than reacting.",
        "when_useful": "Conflict, frustration, urges to send a message you might regret, emotional flooding.",
        "source": "Linehan, M. (2014). DBT Skills Training Manual (2nd ed.). Guilford Press.",
        "disclaimer_level": "low",
    },
    {
        "id": "mind_008",
        "category": "Mindfulness",
        "title": "Loving-Kindness Meditation",
        "tags": ["self-compassion", "anger", "isolation", "depression", "forgiveness"],
        "description": "Loving-kindness meditation cultivates warmth toward yourself and others by silently repeating well-wishing phrases. Research suggests it reduces self-criticism and increases positive emotions over weeks of practice.",
        "how_to": "Sit comfortably. Silently repeat: 'May I be safe. May I be happy. May I be healthy. May I live with ease.' Spend a few minutes here. Then extend the same phrases to: someone you love; a neutral person; someone difficult; all beings. If the phrases feel hollow, that's normal — keep going.",
        "when_useful": "Self-criticism, isolation, lingering anger, or daily practice.",
        "source": "Salzberg, S. (1995). Lovingkindness: The Revolutionary Art of Happiness. Shambhala.",
        "disclaimer_level": "low",
    },
]


# ============================================================================
# PSYCHOEDUCATION
# ============================================================================
# All entries here are general-information only and explicitly recommend
# professional consultation. No diagnostic checklists.
psychoed_entries = [
    {
        "id": "psyed_001",
        "category": "Psychoeducation",
        "title": "Understanding Generalized Anxiety",
        "tags": ["anxiety", "worry", "generalized anxiety", "GAD"],
        "description": "Generalized anxiety involves persistent, excessive worry across multiple areas of life — health, work, relationships, finances — that feels difficult to control. It is often accompanied by physical signs: muscle tension, restlessness, fatigue, sleep disturbance, and difficulty concentrating. Unlike everyday worry, it tends to persist for months and interferes with daily functioning.",
        "how_to": "Effective treatments are well-established and include cognitive behavioral therapy (especially for worry), acceptance-based approaches, and sometimes medication. Self-help techniques like worry time, reframing, and regular exercise can reduce symptoms. A mental health professional can help determine whether what you're experiencing is GAD and what treatment fits.",
        "when_useful": "If worry feels constant, exhausting, and out of proportion to actual circumstances.",
        "source": "American Psychiatric Association (2022). DSM-5-TR; Beyond Blue (beyondblue.org.au).",
        "disclaimer_level": "high",
    },
    {
        "id": "psyed_002",
        "category": "Psychoeducation",
        "title": "Understanding Major Depression",
        "tags": ["depression", "low mood", "anhedonia", "MDD"],
        "description": "Major depression is more than feeling sad — it involves persistent low mood or loss of interest in things that previously felt rewarding, lasting at least two weeks, alongside changes in sleep, appetite, energy, concentration, or self-worth. It is one of the most common mental health conditions and is highly treatable.",
        "how_to": "Evidence-based treatments include psychotherapy (CBT, behavioral activation, interpersonal therapy), medication, and combinations of both. Lifestyle factors — sleep, exercise, social contact — play a meaningful role. Recovery is often gradual and non-linear; small steps count. Speaking with a GP or mental health professional is the most reliable first step.",
        "when_useful": "If low mood persists for weeks and interferes with work, relationships, or self-care.",
        "source": "American Psychiatric Association (2022). DSM-5-TR; Black Dog Institute (blackdoginstitute.org.au).",
        "disclaimer_level": "high",
    },
    {
        "id": "psyed_003",
        "category": "Psychoeducation",
        "title": "Understanding Panic Attacks",
        "tags": ["panic", "anxiety", "panic attack", "panic disorder"],
        "description": "A panic attack is a sudden surge of intense fear or discomfort, peaking within minutes, with physical symptoms like racing heart, shortness of breath, dizziness, sweating, and a sense of unreality or impending doom. They are frightening but not physically dangerous. Many people experience one in their lifetime; some develop panic disorder, where the fear of future attacks itself becomes a problem.",
        "how_to": "During an attack, slow breathing and grounding can help; the attack will peak and pass on its own. Cognitive behavioral therapy for panic, including interoceptive exposure (gradually facing feared body sensations), is highly effective. If panic attacks recur or you start avoiding situations because of them, professional support is recommended.",
        "when_useful": "If you experience recurring sudden fear surges or are avoiding places out of fear of having one.",
        "source": "American Psychiatric Association (2022). DSM-5-TR; Anxiety Recovery Centre Victoria.",
        "disclaimer_level": "high",
    },
    {
        "id": "psyed_004",
        "category": "Psychoeducation",
        "title": "Understanding Post-Traumatic Stress",
        "tags": ["trauma", "PTSD", "flashback", "hypervigilance", "dissociation"],
        "description": "Post-traumatic stress can develop after experiencing or witnessing a deeply distressing event. Common features include unwanted memories or flashbacks, avoidance of reminders, persistent negative mood or beliefs, and being on edge or easily startled. Reactions are normal in the weeks after a trauma; when they persist for months and disrupt daily life, professional help is important.",
        "how_to": "Evidence-based therapies include trauma-focused CBT, prolonged exposure, and EMDR. Trauma work is best done with a trained therapist — self-directed exposure to traumatic memories without support can re-traumatize. Stabilization skills (grounding, self-compassion, sleep) are an important first phase.",
        "when_useful": "If trauma symptoms persist beyond a month and interfere with daily life.",
        "source": "American Psychiatric Association (2022). DSM-5-TR; Phoenix Australia (phoenixaustralia.org).",
        "disclaimer_level": "high",
    },
    {
        "id": "psyed_005",
        "category": "Psychoeducation",
        "title": "Understanding Social Anxiety",
        "tags": ["social anxiety", "shyness", "public speaking", "avoidance"],
        "description": "Social anxiety involves intense fear of being judged or scrutinized in social or performance situations, often leading to avoidance. It goes beyond shyness — the fear is persistent, distressing, and shapes major life choices. Many people with social anxiety recognize the fear is excessive but find it difficult to override.",
        "how_to": "Cognitive behavioral therapy with gradual exposure to feared social situations is the most effective treatment. Working on the underlying belief that you will be negatively evaluated — and testing that belief through small experiments — is central. Medication can help in some cases.",
        "when_useful": "If social fear is causing you to avoid situations you'd otherwise want to engage in.",
        "source": "Clark, D. & Wells, A. (1995). A cognitive model of social phobia. In Heimberg et al., Social Phobia: Diagnosis, Assessment, and Treatment. Guilford.",
        "disclaimer_level": "high",
    },
    {
        "id": "psyed_006",
        "category": "Psychoeducation",
        "title": "Understanding Insomnia",
        "tags": ["insomnia", "sleep", "sleeplessness"],
        "description": "Insomnia involves difficulty falling asleep, staying asleep, or waking too early, with daytime consequences, persisting for at least three nights a week over months. It often becomes self-perpetuating: poor sleep creates anxiety about sleep, which further disrupts sleep. Addressing the cycle, not just the symptom, is what helps.",
        "how_to": "Cognitive behavioral therapy for insomnia (CBT-I) is more effective long-term than sleep medication. Core components include sleep restriction, stimulus control (bed for sleep only), and cognitive work on sleep-related anxiety. Sleep hygiene helps but is rarely sufficient alone for chronic insomnia.",
        "when_useful": "If sleep difficulty has lasted more than a few weeks and affects daytime functioning.",
        "source": "Morin, C. & Espie, C. (2003). Insomnia: A Clinical Guide to Assessment and Treatment. Springer.",
        "disclaimer_level": "medium",
    },
    {
        "id": "psyed_007",
        "category": "Psychoeducation",
        "title": "Understanding Grief",
        "tags": ["grief", "loss", "bereavement", "mourning"],
        "description": "Grief is the natural response to loss. It is not linear — waves of sadness, anger, numbness, longing, and even moments of relief can come in any order. There is no fixed timeline. Most people gradually integrate the loss over months and years; for some, grief becomes prolonged and disabling, in which case specific support is helpful.",
        "how_to": "Allow grief to take the form it takes. Maintain basic routines (sleep, food, gentle movement). Speak about the person you lost — silence often deepens isolation. Consider grief support groups; shared loss can feel less alone. If grief feels stuck or you can't function months later, a therapist trained in grief can help.",
        "when_useful": "After any significant loss — death, relationship, role, or identity.",
        "source": "Worden, J. (2018). Grief Counseling and Grief Therapy (5th ed.). Springer.",
        "disclaimer_level": "medium",
    },
    {
        "id": "psyed_008",
        "category": "Psychoeducation",
        "title": "Understanding OCD",
        "tags": ["OCD", "obsessions", "compulsions", "intrusive thoughts"],
        "description": "Obsessive-compulsive disorder involves intrusive, unwanted thoughts (obsessions) that cause anxiety, paired with behaviors or mental acts (compulsions) performed to reduce that anxiety. The compulsions provide brief relief but reinforce the cycle. OCD is treatable, and crucially, the content of the obsessions does not reflect the person's character.",
        "how_to": "Exposure and response prevention (ERP) is the gold-standard treatment: gradually facing feared thoughts or situations without performing the compulsion. ERP is hard but effective. Medication can be helpful, often alongside therapy. Self-directed ERP for severe OCD is generally not recommended — work with a trained therapist.",
        "when_useful": "If intrusive thoughts and rituals are eating significant time and distress.",
        "source": "Foa, E., Yadin, E., & Lichner, T. (2012). Exposure and Response (Ritual) Prevention for OCD. Oxford University Press.",
        "disclaimer_level": "high",
    },
    {
        "id": "psyed_009",
        "category": "Psychoeducation",
        "title": "When to Seek Professional Help",
        "tags": ["help-seeking", "therapy", "professional support"],
        "description": "There is no minimum threshold of suffering required to talk to a professional. Some signs it might be time: symptoms have lasted more than a few weeks, daily functioning is affected (work, relationships, self-care), self-help isn't shifting things, or you're using substances to cope. Seeking help is a practical decision, not an admission of failure.",
        "how_to": "In Australia: see a GP for a Mental Health Treatment Plan, which gives subsidized access to a psychologist. Beyond Blue (1300 22 4636) and Lifeline (13 11 14) offer phone support. Headspace serves people aged 12-25. Online directories like the Australian Psychological Society's 'Find a Psychologist' list practitioners by specialty and location.",
        "when_useful": "Whenever you're considering it. Earlier is generally better than later.",
        "source": "Beyond Blue (beyondblue.org.au); Australian Psychological Society (psychology.org.au).",
        "disclaimer_level": "medium",
    },
    {
        "id": "psyed_010",
        "category": "Psychoeducation",
        "title": "Crisis Resources (Australia)",
        "tags": ["crisis", "suicide", "emergency", "lifeline", "Australia"],
        "description": "If you are in immediate danger, call 000. For mental health crisis support in Australia: Lifeline (13 11 14, 24/7), Suicide Call Back Service (1300 659 467), Beyond Blue (1300 22 4636), Kids Helpline (1800 55 1800, ages 5-25), 13YARN (13 92 76, for Aboriginal and Torres Strait Islander people). These services are free, confidential, and staffed by trained counsellors.",
        "how_to": "You don't need to be 'in crisis enough' to call. If you're not sure whether to call, that's a sign it's worth calling. You can also visit a hospital emergency department, which has mental health staff. Bringing someone you trust with you can make it easier.",
        "when_useful": "Suicidal thoughts, self-harm urges, acute distress, or supporting someone in crisis.",
        "source": "Lifeline Australia (lifeline.org.au); Beyond Blue (beyondblue.org.au).",
        "disclaimer_level": "high",
    },
]


# ============================================================================
# Combine + write
# ============================================================================
all_entries = cbt_entries + mindfulness_entries + psychoed_entries

# Sanity check: no duplicate IDs
ids = [e["id"] for e in all_entries]
assert len(ids) == len(set(ids)), "Duplicate IDs in knowledge base!"

print(f"\n📊 Total entries: {len(all_entries)}")
print(f"   CBT:             {len(cbt_entries)}")
print(f"   Mindfulness:     {len(mindfulness_entries)}")
print(f"   Psychoeducation: {len(psychoed_entries)}")

with open(KB_PATH, "w", encoding="utf-8") as f:
    json.dump(all_entries, f, indent=2, ensure_ascii=False)

print(f"\n✅ Wrote {len(all_entries)} entries to {KB_PATH}")
print(f"   File size: {KB_PATH.stat().st_size / 1024:.1f} KB")

In [ ]:
# ============================================================================
# Cell 2 — Embed Knowledge Base into Chroma
#
# Reads data/external/knowledge_base.json, embeds each entry, and stores in
# a SEPARATE Chroma collection ('knowledge_base') in the SAME persistent
# DB at models/chroma_db/.
#
# Two collections coexist in the same DB:
#   - counsel_chat    (Counsel Chat questions, ~863 items)
#   - knowledge_base  (curated CBT/mindfulness/psychoed, ~30 items)
#
# What gets embedded per entry: title + description + tags.
# `how_to` stays in metadata — it's instructions, not a search target.
#
# Set FORCE_REBUILD = True (default) to wipe and re-index. Set False for
# idempotent reruns once you're happy with the index.
# ============================================================================

import json
from pathlib import Path

import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer

# --- Locate project root -----------------------------------------------------
_cwd = Path.cwd().resolve()
PROJ_ROOT = next(
    (p for p in [_cwd, *_cwd.parents]
     if (p / "data").exists() or (p / ".env").exists()),
    _cwd,
)
KB_PATH    = PROJ_ROOT / "data" / "external" / "knowledge_base.json"
CHROMA_DIR = PROJ_ROOT / "models" / "chroma_db"

EMBED_MODEL_NAME = "all-MiniLM-L6-v2"
COLLECTION_NAME  = "knowledge_base"
BATCH_SIZE       = 32
FORCE_REBUILD    = True   # ← True for first build / rebuild; False for idempotent reruns

print(f"📁 KB JSON:    {KB_PATH}")
print(f"📂 Chroma dir: {CHROMA_DIR}")


# --- Load JSON ---------------------------------------------------------------
if not KB_PATH.exists():
    raise FileNotFoundError(
        f"{KB_PATH} not found. Run the knowledge-base build cell first."
    )

with open(KB_PATH, "r", encoding="utf-8") as f:
    entries = json.load(f)

print(f"\n📊 Loaded {len(entries)} entries from JSON")
counts = {}
for e in entries:
    counts[e["category"]] = counts.get(e["category"], 0) + 1
for cat, n in counts.items():
    print(f"   {cat:18s} {n}")


# --- Init Chroma client ------------------------------------------------------
client = chromadb.PersistentClient(
    path=str(CHROMA_DIR),
    settings=Settings(anonymized_telemetry=False),
)

existing = [c.name for c in client.list_collections()]

if COLLECTION_NAME in existing and not FORCE_REBUILD:
    coll = client.get_collection(COLLECTION_NAME)
    print(
        f"\n⏭️  Collection '{COLLECTION_NAME}' already exists with "
        f"{coll.count()} items — skipping."
    )
else:
    if COLLECTION_NAME in existing:
        print(f"\n🗑️  Deleting existing '{COLLECTION_NAME}' collection ...")
        client.delete_collection(COLLECTION_NAME)

    coll = client.create_collection(
        name=COLLECTION_NAME,
        metadata={"hnsw:space": "cosine"},   # match counsel_chat collection
    )

    # --- Build the searchable text per entry --------------------------------
    # title carries strong signal; description is the semantic core; tags
    # act as keyword anchors that improve recall on tag-shaped queries.
    def to_search_text(e):
        tags = ", ".join(e.get("tags", []))
        return f"{e['title']}. {e['description']} Tags: {tags}"

    search_texts = [to_search_text(e) for e in entries]

    # --- Embed --------------------------------------------------------------
    print(f"\n🧠 Loading embedding model '{EMBED_MODEL_NAME}' ...")
    embed_model = SentenceTransformer(EMBED_MODEL_NAME)

    print(f"🔢 Embedding {len(search_texts)} entries ...")
    embeddings = embed_model.encode(
        search_texts,
        batch_size=BATCH_SIZE,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

    # --- Build metadata payload ---------------------------------------------
    # Chroma metadata must be primitive types; we serialize tags to a string.
    ids = [e["id"] for e in entries]
    documents = search_texts   # store the embedded text as the document too
    metadatas = [
        {
            "source":           "knowledge_base",
            "category":         e["category"],
            "title":            e["title"],
            "tags":             ", ".join(e.get("tags", [])),
            "description":      e["description"],
            "how_to":           e["how_to"],
            "when_useful":      e["when_useful"],
            "citation":         e["source"],
            "disclaimer_level": e["disclaimer_level"],
        }
        for e in entries
    ]

    # --- Insert -------------------------------------------------------------
    print(f"💾 Inserting into Chroma in batches of {BATCH_SIZE} ...")
    for start in range(0, len(ids), BATCH_SIZE):
        end = start + BATCH_SIZE
        coll.add(
            ids=ids[start:end],
            documents=documents[start:end],
            embeddings=embeddings[start:end].tolist(),
            metadatas=metadatas[start:end],
        )

    print(f"\n✅ Indexed {coll.count()} entries into '{COLLECTION_NAME}'.")
    print(f"   Persisted to {CHROMA_DIR}")


# --- Show both collections in the DB ----------------------------------------
print("\n📚 All collections in the DB:")
for c in client.list_collections():
    info = client.get_collection(c.name)
    print(f"   - {c.name:20s} {info.count():,} items")

In [ ]:
# ============================================================================
# Cell 3 — Unified Retrieval (counsel_chat + knowledge_base)
#
# Defines `retrieve(query, k_counsel, k_kb)` that queries BOTH collections
# in the Chroma DB and returns a merged, normalized list of results.
#
# Design choices:
#   - Separate k per collection (default 3 + 2): counsel_chat for empathic
#     "what others have said," knowledge_base for actionable techniques.
#   - Bucketed merge, NOT a global sort by similarity — KB entries are
#     shorter and tend to score lower even when more useful.
#   - Normalized schema across sources so downstream LLM code doesn't have
#     to branch on `source`.
#
# This is the function the LLM step will call.
# ============================================================================

from pathlib import Path
from typing import Optional

import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer

# --- Locate project root -----------------------------------------------------
_cwd = Path.cwd().resolve()
PROJ_ROOT = next(
    (p for p in [_cwd, *_cwd.parents]
     if (p / "data").exists() or (p / ".env").exists()),
    _cwd,
)
CHROMA_DIR = PROJ_ROOT / "models" / "chroma_db"

EMBED_MODEL_NAME       = "all-MiniLM-L6-v2"
COUNSEL_COLLECTION     = "counsel_chat"
KB_COLLECTION          = "knowledge_base"
DEFAULT_K_COUNSEL      = 3
DEFAULT_K_KB           = 2


# --- One-time init -----------------------------------------------------------
print(f"🧠 Loading embedding model '{EMBED_MODEL_NAME}' ...")
_embed_model = SentenceTransformer(EMBED_MODEL_NAME)

_client = chromadb.PersistentClient(
    path=str(CHROMA_DIR),
    settings=Settings(anonymized_telemetry=False),
)
_counsel_coll = _client.get_collection(COUNSEL_COLLECTION)
_kb_coll      = _client.get_collection(KB_COLLECTION)
print(f"✅ Connected | counsel_chat: {_counsel_coll.count():,} | "
      f"knowledge_base: {_kb_coll.count()}\n")


# --- Helpers -----------------------------------------------------------------
def _embed(query: str) -> list:
    """Embed a single query string."""
    return _embed_model.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True,
    )[0].tolist()


def _query_counsel(q_emb, k: int) -> list[dict]:
    """Query the counsel_chat collection and normalize results."""
    if k <= 0:
        return []
    res = _counsel_coll.query(query_embeddings=[q_emb], n_results=k)
    out = []
    for doc, meta, dist in zip(
        res["documents"][0],
        res["metadatas"][0],
        res["distances"][0],
    ):
        out.append({
            "source":            "counsel_chat",
            "similarity":        round(1.0 - dist, 4),
            "topic_or_category": meta.get("topic", "unknown"),
            # `text` is what was embedded (the question)
            "text":              doc,
            # `content` is what the LLM should actually see — the therapist answer(s)
            "content":           meta.get("answerText", ""),
            # extras for transparency / citation
            "raw_meta":          meta,
        })
    return out


def _query_kb(q_emb, k: int) -> list[dict]:
    """Query the knowledge_base collection and normalize results."""
    if k <= 0:
        return []
    res = _kb_coll.query(query_embeddings=[q_emb], n_results=k)
    out = []
    for doc, meta, dist in zip(
        res["documents"][0],
        res["metadatas"][0],
        res["distances"][0],
    ):
        # Compose the LLM-visible content from KB metadata
        content = (
            f"{meta.get('description', '')}\n\n"
            f"How to use: {meta.get('how_to', '')}\n\n"
            f"When useful: {meta.get('when_useful', '')}"
        )
        out.append({
            "source":            "knowledge_base",
            "similarity":        round(1.0 - dist, 4),
            "topic_or_category": meta.get("category", "unknown"),
            "text":              meta.get("title", doc),
            "content":           content,
            "raw_meta":          meta,
        })
    return out


# --- Main retrieval function -------------------------------------------------
def retrieve(
    query: str,
    k_counsel: int = DEFAULT_K_COUNSEL,
    k_kb: int = DEFAULT_K_KB,
) -> list[dict]:
    """
    Retrieve relevant chunks from both Chroma collections.

    Parameters
    ----------
    query : str
        The user message (already cleaned).
    k_counsel : int
        Number of counsel_chat results.
    k_kb : int
        Number of knowledge_base results.

    Returns
    -------
    list of dicts with keys:
        source, similarity, topic_or_category, text, content, raw_meta

    Counsel chat results come first (for empathic grounding), then KB
    results (for actionable techniques). Within each bucket, results are
    ordered by similarity (highest first).
    """
    q_emb = _embed(query)
    counsel_results = _query_counsel(q_emb, k_counsel)
    kb_results      = _query_kb(q_emb, k_kb)
    # Bucketed merge: counsel first, then KB. NOT sorted globally by similarity.
    return counsel_results + kb_results


# --- Sanity check ------------------------------------------------------------
def _print_results(query: str, results: list[dict]) -> None:
    print("=" * 80)
    print(f"🔍 Query: {query}")
    print("=" * 80)
    for i, r in enumerate(results, 1):
        src_icon = "💬" if r["source"] == "counsel_chat" else "📘"
        print(
            f"\n  [{i}] {src_icon} {r['source']:14s} "
            f"sim={r['similarity']:.3f}  "
            f"{r['topic_or_category']}"
        )
        print(f"      • {r['text'][:140]}")
        print(f"      → {r['content'][:200].replace(chr(10), ' ')} ...")
    print()


test_queries = [
    "I've been feeling really anxious lately and don't know what to do",
    "How can I stop catastrophizing every small problem?",
    "I can't sleep at night, my mind keeps racing",
    "I just lost my mother and don't know how to cope",
    "I want to feel less depressed but have no motivation to do anything",
]

for q in test_queries:
    results = retrieve(q, k_counsel=3, k_kb=2)
    _print_results(q, results)

## 7.3 Step 3 — LLM response generation (using free gemini api key in .env file)

In [ ]:
# ============================================================================
# Cell A — Gemini API Setup & Smoke Test
#
# Verifies GEMINI_API_KEY is in .env, initializes the genai client, and runs
# a one-line smoke test to confirm credentials work BEFORE we wire it into
# the pipeline.
#
# Requires: pip install google-genai python-dotenv
# (Note: google-genai is the new SDK. The old google-generativeai is
#  deprecated as of 2025/2026.)
# ============================================================================

import os
import sys
from pathlib import Path

from dotenv import load_dotenv
from google import genai

# --- Locate project root -----------------------------------------------------
_cwd = Path.cwd().resolve()
PROJ_ROOT = next(
    (p for p in [_cwd, *_cwd.parents]
     if (p / "data").exists() or (p / ".env").exists()),
    _cwd,
)
ENV_PATH = PROJ_ROOT / ".env"

# --- Load .env ---------------------------------------------------------------
if not ENV_PATH.exists():
    print(f"❌ .env not found at {ENV_PATH}")
    print("   Create it with: GEMINI_API_KEY=your_key_here")
    sys.exit(1)

load_dotenv(ENV_PATH)

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
if not GEMINI_API_KEY:
    print(f"❌ GEMINI_API_KEY not found in {ENV_PATH}")
    print("   Add this line to your .env:")
    print("       GEMINI_API_KEY=your_key_here")
    print("   Get a key from: https://aistudio.google.com/apikey")
    sys.exit(1)

# Don't print the full key, but confirm it loaded
print(f"✅ Loaded GEMINI_API_KEY from {ENV_PATH.name} "
      f"(starts with '{GEMINI_API_KEY[:6]}...', length {len(GEMINI_API_KEY)})")


# --- Init client + smoke test -----------------------------------------------
GEMINI_MODEL = "gemini-2.5-flash"   # fast, cheap, plenty good for empathic chat

print(f"\n🤖 Initializing Gemini client (model: {GEMINI_MODEL}) ...")
gemini_client = genai.Client(api_key=GEMINI_API_KEY)

print("🩺 Running smoke test ...")
try:
    resp = gemini_client.models.generate_content(
        model=GEMINI_MODEL,
        contents="Reply with exactly the three words: hello from gemini",
    )
    print(f"\n✅ Smoke test response:\n   {resp.text.strip()}")
except Exception as e:
    print(f"\n❌ Smoke test failed: {type(e).__name__}: {e}")
    print("   Common causes:")
    print("     - Invalid API key")
    print("     - Key not enabled for Gemini API")
    print("     - Network/firewall blocking generativelanguage.googleapis.com")
    raise

print("\n🎉 Gemini client ready. Move on to Cell B (prompt assembly).")

In [ ]:
# ============================================================================
# Cell B — System Prompt & Prompt Assembly
#
# Defines the system prompt that governs Claude/Gemini's tone and safety
# behavior, and a build_prompt() helper that turns retrieved chunks +
# conversation history + user message into the structured payload Gemini
# will see.
#
# This cell does NOT call the Gemini API. It just builds strings. That way
# we can iterate on the prompt cheaply, then call in Cell C.
# ============================================================================

from textwrap import dedent

# ---------------------------------------------------------------------------
# System prompt
# ---------------------------------------------------------------------------
# Design notes:
#   - Sets identity: empathic peer-support, not therapist, not diagnostician.
#   - Hard rules use imperative voice ("Never ...") — clearer to instruction-
#     tuned models than soft suggestions.
#   - The grounding rule ("draw on the provided context, but don't fabricate
#     citations") protects against confabulated source IDs.
#   - Australian crisis resources are listed verbatim so the model has them
#     correctly even on cold queries; the crisis classifier handles the
#     critical path separately, but this is a defense-in-depth backup.
#   - Length guidance: most replies should be ~3-6 short paragraphs. Long
#     replies overwhelm someone in distress; one-liners feel dismissive.
#
# To tune: edit this string and re-run the cell. Test changes with Cell C.
# ---------------------------------------------------------------------------
MENTAL_HEALTH_SYSTEM_PROMPT = dedent("""
    You are a warm, careful mental health support companion. You are NOT a
    therapist, doctor, or diagnostician — you are a thoughtful presence that
    helps people feel heard and offers evidence-informed self-help ideas.

    HOW TO RESPOND
    - Lead with acknowledgement. Reflect what the person is feeling in your
      own words before offering anything else. People want to be heard first.
    - Use plain, gentle language. Short sentences. Avoid clinical jargon
      unless explaining it.
    - Offer one or two concrete, doable suggestions — not a list of ten.
      Pick the most relevant from the provided context.
    - Always close by inviting the person to share more, or to consider
      professional support if the situation warrants it.
    - Keep replies to roughly 3 to 6 short paragraphs unless the person
      asks for more detail.

    GROUNDING & CITATIONS
    - You will be given retrieved context labeled [S1], [S2], etc. Draw on
      this context when offering techniques or framing.
    - When you use information from a source, cite it inline like (S1) or
      (S2). Do not invent source IDs that weren't provided.
    - If the context doesn't fit the user's situation, rely on general
      empathic listening rather than forcing a poor match.

    WHAT YOU MUST NOT DO
    - Do not diagnose. Do not say "you have X" or "this sounds like X
      disorder." You can say "what you're describing is something many
      people experience" or "a professional could help you understand
      what's going on."
    - Do not prescribe or recommend specific medications or dosages.
    - Do not promise outcomes ("you will feel better in a week").
    - Do not minimize ("it could be worse," "others have it harder").
    - Do not push religion, politics, or unsolicited life philosophy.
    - Do not pretend to be human. If asked directly, say you're an AI
      support tool.

    SAFETY
    - If the person mentions suicidal thoughts, self-harm, or being in
      immediate danger, gently acknowledge what they shared and provide
      Australian crisis resources:
        • Lifeline: 13 11 14 (24/7)
        • Beyond Blue: 1300 22 4636
        • Suicide Call Back Service: 1300 659 467
        • In immediate danger: call 000
    - Encourage professional support for ongoing or severe distress.

    TONE EXAMPLES
    - Good: "That sounds really heavy. It makes sense you'd feel exhausted
      carrying it."
    - Bad: "I understand. Have you tried mindfulness?" (too quick to advice)
    - Bad: "Based on your symptoms, you may have generalized anxiety
      disorder." (diagnosing)
""").strip()


# ---------------------------------------------------------------------------
# Conversation history config
# ---------------------------------------------------------------------------
MAX_HISTORY_TURNS = 6   # last 6 turns (3 user + 3 assistant) max


# ---------------------------------------------------------------------------
# Prompt assembly
# ---------------------------------------------------------------------------
def _format_chunk(idx: int, chunk: dict) -> str:
    """Format one retrieved chunk as a labeled source block."""
    source = chunk["source"]
    topic  = chunk["topic_or_category"]
    text   = chunk["text"]
    body   = chunk["content"]

    # Truncate very long bodies (some Counsel Chat answers are 1000+ chars)
    # to keep the prompt focused. 800 chars is roughly 200 tokens.
    if len(body) > 800:
        body = body[:800].rstrip() + " ..."

    if source == "counsel_chat":
        header = f"[S{idx}] (Counsel Chat — topic: {topic})"
        return f"{header}\nQuestion: {text}\nTherapist response: {body}"
    else:  # knowledge_base
        header = f"[S{idx}] (Knowledge Base — {topic}: {text})"
        return f"{header}\n{body}"


def _format_history(history: list[dict]) -> str:
    """Format last N turns of conversation. history is a list of
    {'role': 'user'|'assistant', 'content': str}."""
    if not history:
        return "(No prior conversation.)"
    recent = history[-MAX_HISTORY_TURNS:]
    lines = []
    for turn in recent:
        role = "User" if turn["role"] == "user" else "Assistant"
        lines.append(f"{role}: {turn['content']}")
    return "\n".join(lines)


def build_prompt(
    user_message: str,
    retrieved_chunks: list[dict],
    history: list[dict] = None,
) -> str:
    """
    Assemble the user-side payload (the system prompt is passed separately
    to Gemini's generate_content as system_instruction).

    Returns a single string with three labeled sections:
        RETRIEVED CONTEXT, RECENT CONVERSATION, USER MESSAGE.
    """
    if history is None:
        history = []

    if retrieved_chunks:
        context_blocks = "\n\n".join(
            _format_chunk(i + 1, c) for i, c in enumerate(retrieved_chunks)
        )
    else:
        context_blocks = "(No retrieved context available.)"

    history_block = _format_history(history)

    return dedent(f"""
        RETRIEVED CONTEXT (use [S1], [S2]... to cite):
        {context_blocks}

        ---
        RECENT CONVERSATION:
        {history_block}

        ---
        CURRENT USER MESSAGE:
        {user_message}

        Respond as the support companion described in your system instructions.
    """).strip()


# ---------------------------------------------------------------------------
# Sanity check — eyeball the assembled prompt for one query
# ---------------------------------------------------------------------------
# This uses retrieve() from Cell 3 (unified retrieval). Make sure that cell
# has been run first.
test_user_msg = "I've been feeling really anxious lately and don't know what to do"
test_history = [
    {"role": "user",      "content": "Hi, can I talk to you?"},
    {"role": "assistant", "content": "Of course. I'm here. What's on your mind?"},
]

retrieved = retrieve(test_user_msg, k_counsel=3, k_kb=2)
prompt = build_prompt(test_user_msg, retrieved, test_history)

print("=" * 80)
print("SYSTEM PROMPT")
print("=" * 80)
print(MENTAL_HEALTH_SYSTEM_PROMPT)
print(f"\n   ({len(MENTAL_HEALTH_SYSTEM_PROMPT)} chars / "
      f"~{len(MENTAL_HEALTH_SYSTEM_PROMPT) // 4} tokens)")

print("\n" + "=" * 80)
print("ASSEMBLED USER-SIDE PROMPT")
print("=" * 80)
print(prompt)
print(f"\n   ({len(prompt)} chars / ~{len(prompt) // 4} tokens)")

In [ ]:
# ============================================================================
# Cell B (patch) — Fix build_prompt indentation
#
# Replaces build_prompt() so the assembled prompt prints cleanly without the
# leading whitespace that dedent + f-string interpolation creates. Re-run
# this cell after Cell B; everything else (system prompt, _format_chunk,
# _format_history, MAX_HISTORY_TURNS) stays as-is.
# ============================================================================

def build_prompt(
    user_message: str,
    retrieved_chunks: list[dict],
    history: list[dict] = None,
) -> str:
    """
    Assemble the user-side payload (the system prompt is passed separately
    to Gemini's generate_content as system_instruction).

    Returns a single string with three labeled sections:
        RETRIEVED CONTEXT, RECENT CONVERSATION, USER MESSAGE.
    """
    if history is None:
        history = []

    if retrieved_chunks:
        context_blocks = "\n\n".join(
            _format_chunk(i + 1, c) for i, c in enumerate(retrieved_chunks)
        )
    else:
        context_blocks = "(No retrieved context available.)"

    history_block = _format_history(history)

    parts = [
        "RETRIEVED CONTEXT (use [S1], [S2]... to cite):",
        context_blocks,
        "",
        "---",
        "RECENT CONVERSATION:",
        history_block,
        "",
        "---",
        "CURRENT USER MESSAGE:",
        user_message,
        "",
        "Respond as the support companion described in your system instructions.",
    ]
    return "\n".join(parts)


# --- Re-run the sanity check ------------------------------------------------
test_user_msg = "I've been feeling really anxious lately and don't know what to do"
test_history = [
    {"role": "user",      "content": "Hi, can I talk to you?"},
    {"role": "assistant", "content": "Of course. I'm here. What's on your mind?"},
]

retrieved = retrieve(test_user_msg, k_counsel=3, k_kb=2)
prompt = build_prompt(test_user_msg, retrieved, test_history)

print("=" * 80)
print("ASSEMBLED USER-SIDE PROMPT (clean version)")
print("=" * 80)
print(prompt)
print(f"\n   ({len(prompt)} chars / ~{len(prompt) // 4} tokens)")

In [ ]:

# ============================================================================
# Cell C — End-to-End Response Generation
#
# Defines generate_response(user_msg, history) which is the full RAG-LLM
# path: retrieve → build prompt → call Gemini → return packaged response.
#
# Then runs 4 representative messages through it so you can read what the
# chatbot actually says.
#
# Requires (already loaded by previous cells):
#   - retrieve()                          (Cell 3)
#   - MENTAL_HEALTH_SYSTEM_PROMPT         (Cell B)
#   - build_prompt()                      (Cell B + patch)
#   - gemini_client, GEMINI_MODEL         (Cell A)
# ============================================================================

import re
from google.genai import types

# --- Generation config -------------------------------------------------------
GEN_TEMPERATURE      = 0.7   # warm but not erratic
GEN_MAX_OUTPUT_TOKENS = 600  # ~3-6 short paragraphs


# --- Source extraction helper -----------------------------------------------
def _extract_cited_sources(reply_text: str, retrieved: list[dict]) -> list[dict]:
    """
    Find (S1), (S2)... citations in the reply and return the matching
    retrieved chunks (deduped, in citation order).
    """
    cited_ids = []
    for m in re.finditer(r"\(S(\d+)\)", reply_text):
        idx = int(m.group(1))
        if 1 <= idx <= len(retrieved) and idx not in cited_ids:
            cited_ids.append(idx)
    return [retrieved[i - 1] for i in cited_ids]


# --- Main entry point --------------------------------------------------------
def generate_response(
    user_message: str,
    history: list[dict] = None,
    k_counsel: int = 3,
    k_kb: int = 2,
) -> dict:
    """
    Run the full retrieval + generation pipeline for one user message.

    Returns
    -------
    dict with:
        text        : str    — the chatbot's reply
        sources     : list   — chunks the model actually cited inline
        retrieved   : list   — all retrieved chunks (for transparency)
        prompt_used : str    — the assembled user-side prompt (for debug)
    """
    if history is None:
        history = []

    # 1. Retrieve
    retrieved = retrieve(user_message, k_counsel=k_counsel, k_kb=k_kb)

    # 2. Build prompt
    user_prompt = build_prompt(user_message, retrieved, history)

    # 3. Call Gemini
    config = types.GenerateContentConfig(
        system_instruction=MENTAL_HEALTH_SYSTEM_PROMPT,
        temperature=GEN_TEMPERATURE,
        max_output_tokens=GEN_MAX_OUTPUT_TOKENS,
    )
    response = gemini_client.models.generate_content(
        model=GEMINI_MODEL,
        contents=user_prompt,
        config=config,
    )
    reply_text = (response.text or "").strip()

    # 4. Extract which sources the model actually cited
    cited = _extract_cited_sources(reply_text, retrieved)

    return {
        "text":        reply_text,
        "sources":     cited,
        "retrieved":   retrieved,
        "prompt_used": user_prompt,
    }


# --- Smoke test on representative messages ----------------------------------
test_messages = [
    "I've been feeling really anxious lately and don't know what to do",
    "I want to feel less depressed but have no motivation to do anything",
    "My partner and I keep fighting and I don't know if we should stay together",
    "I can't sleep at night, my mind keeps racing and I'm exhausted",
]

for i, msg in enumerate(test_messages, 1):
    print("\n" + "=" * 80)
    print(f"TEST {i}: {msg}")
    print("=" * 80)

    result = generate_response(msg, history=[])

    print(f"\n💬 REPLY:\n{result['text']}\n")

    if result["sources"]:
        print(f"📎 CITED SOURCES ({len(result['sources'])}):")
        for s in result["sources"]:
            print(f"   - {s['source']:14s} | {s['topic_or_category']:25s} | "
                  f"{s['text'][:80]}")
    else:
        print("📎 CITED SOURCES: (none cited inline)")

    print()

In [ ]:
# ============================================================================
# Cell C (patch) — Disable Gemini thinking to stop truncation
#
# gemini-2.5-flash has "thinking" enabled by default. Thinking tokens count
# against max_output_tokens, so with max_output_tokens=600 the model often
# burns 400+ tokens reasoning internally and runs out of budget before
# finishing the visible reply. That's why some test replies cut off mid-
# sentence in Cell C.
#
# Fix: pass thinking_config=ThinkingConfig(thinking_budget=0). Standard
# recommendation for chat-style UIs.
#
# Re-run this cell after Cell C; only generate_response is replaced.
# ============================================================================

from google.genai import types


def generate_response(
    user_message: str,
    history: list[dict] = None,
    k_counsel: int = 3,
    k_kb: int = 2,
) -> dict:
    """Full retrieval + generation pipeline. Thinking disabled for chat use."""
    if history is None:
        history = []

    retrieved   = retrieve(user_message, k_counsel=k_counsel, k_kb=k_kb)
    user_prompt = build_prompt(user_message, retrieved, history)

    config = types.GenerateContentConfig(
        system_instruction=MENTAL_HEALTH_SYSTEM_PROMPT,
        temperature=GEN_TEMPERATURE,
        max_output_tokens=GEN_MAX_OUTPUT_TOKENS,
        # ↓ The fix. Disable internal "thinking" so the full token budget
        #   goes to the visible reply, not invisible reasoning.
        thinking_config=types.ThinkingConfig(thinking_budget=0),
    )
    response = gemini_client.models.generate_content(
        model=GEMINI_MODEL,
        contents=user_prompt,
        config=config,
    )
    reply_text = (response.text or "").strip()

    cited = _extract_cited_sources(reply_text, retrieved)

    return {
        "text":        reply_text,
        "sources":     cited,
        "retrieved":   retrieved,
        "prompt_used": user_prompt,
        # debug: how the response actually finished + token usage
        "_finish":     str(response.candidates[0].finish_reason)
                       if response.candidates else "?",
        "_usage":      {
            "prompt_tokens":   getattr(response.usage_metadata, "prompt_token_count", None),
            "output_tokens":   getattr(response.usage_metadata, "candidates_token_count", None),
            "thoughts_tokens": getattr(response.usage_metadata, "thoughts_token_count", None),
        },
    }


# --- Re-run smoke test ------------------------------------------------------
test_messages = [
    "I've been feeling really anxious lately and don't know what to do",
    "I want to feel less depressed but have no motivation to do anything",
    "My partner and I keep fighting and I don't know if we should stay together",
    "I can't sleep at night, my mind keeps racing and I'm exhausted",
]

for i, msg in enumerate(test_messages, 1):
    print("\n" + "=" * 80)
    print(f"TEST {i}: {msg}")
    print("=" * 80)

    result = generate_response(msg, history=[])

    print(f"\n💬 REPLY:\n{result['text']}\n")

    if result["sources"]:
        print(f"📎 CITED SOURCES ({len(result['sources'])}):")
        for s in result["sources"]:
            print(f"   - {s['source']:14s} | {s['topic_or_category']:25s} | "
                  f"{s['text'][:80]}")
    else:
        print("📎 CITED SOURCES: (none cited inline)")

    print(f"\n[debug] finish={result['_finish']}  usage={result['_usage']}")

##  7.4 Step 4 — Crisis response branch. When the classifier flags, skip RAG, return empathic acknowledgement + crisis resources, log the event. Orchestration. The single respond(user_message, history) that ties classifier → router → (crisis path | RAG path) → return.

In [ ]:
# ============================================================================
# Cell D — Crisis Branch + Orchestrator
#
# Defines:
#   1. build_crisis_response(user_msg, classifier_result) — empathic
#      acknowledgement (LLM) + hardcoded crisis resources (never LLM).
#   2. _log_crisis_event(...) — appends a JSONL record for safety review.
#   3. respond(user_message, history) — the single entry point. Cleans,
#      classifies, routes to crisis branch or RAG branch, returns a dict
#      in a UNIFIED shape so Streamlit doesn't have to branch.
#
# Requires (already in session):
#   - clean_text()                       (cleaning cell)
#   - is_crisis()                        (predict cell)
#   - retrieve(), generate_response()    (Cells 3, C-fix)
#   - gemini_client, GEMINI_MODEL        (Cell A)
# ============================================================================

import json
from datetime import datetime, timezone
from pathlib import Path

from google.genai import types

# --- Locate project root + log path -----------------------------------------
_cwd = Path.cwd().resolve()
PROJ_ROOT = next(
    (p for p in [_cwd, *_cwd.parents]
     if (p / "data").exists() or (p / ".env").exists()),
    _cwd,
)
CRISIS_LOG_PATH = PROJ_ROOT / "data" / "processed" / "crisis_log.jsonl"
CRISIS_LOG_PATH.parent.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------------------------
# Hardcoded crisis resources block — NEVER LLM-generated. Phone numbers
# must be exact. Update only if the source helplines change.
# ---------------------------------------------------------------------------
CRISIS_RESOURCES_BLOCK = (
    "If you are in immediate danger, please call **000**.\n\n"
    "You don't have to face this alone. These services are free, confidential, "
    "and staffed by trained counsellors right now:\n\n"
    "• **Lifeline** — 13 11 14 (24/7)\n"
    "• **Suicide Call Back Service** — 1300 659 467\n"
    "• **Beyond Blue** — 1300 22 4636\n"
    "• **13YARN** — 13 92 76 (for Aboriginal and Torres Strait Islander people)\n"
    "• **Kids Helpline** — 1800 55 1800 (ages 5–25)\n\n"
    "If you can, please reach out to one of these now. Talking to someone "
    "trained can help you get through this moment."
)

# Short, focused system prompt for the crisis acknowledgement only.
# Deliberately separate from MENTAL_HEALTH_SYSTEM_PROMPT — different job.
CRISIS_SYSTEM_PROMPT = (
    "You are responding to someone who has just expressed thoughts of suicide, "
    "self-harm, or being in serious crisis. Your only job right now is to "
    "acknowledge their pain warmly and personally — NOT to offer techniques, "
    "advice, or solutions. Crisis resources will be appended to your reply "
    "automatically; do NOT include any phone numbers or helpline names in your "
    "response.\n\n"
    "Write 2 to 4 short sentences. Be human and gentle. Reflect what they said "
    "in your own words. Tell them their pain matters and that they are not "
    "alone. Do not say 'I understand' (you don't). Do not minimize. Do not "
    "promise things will be okay. Do not diagnose. End by gently letting them "
    "know that help is available right now."
)


# ---------------------------------------------------------------------------
# Crisis response builder
# ---------------------------------------------------------------------------
def build_crisis_response(
    user_message: str,
    classifier_result: dict,
) -> dict:
    """
    Build a response for a crisis-classified message.
    LLM writes the empathic acknowledgement; resources block is hardcoded.

    Returns a dict in the same shape as generate_response() so the orchestrator
    can return either without the caller branching.
    """
    # --- LLM acknowledgement -------------------------------------------------
    config = types.GenerateContentConfig(
        system_instruction=CRISIS_SYSTEM_PROMPT,
        temperature=0.6,                         # slightly cooler than RAG path
        max_output_tokens=200,                   # short by design
        thinking_config=types.ThinkingConfig(thinking_budget=0),
    )

    try:
        response = gemini_client.models.generate_content(
            model=GEMINI_MODEL,
            contents=f"The person just said: \"{user_message}\"",
            config=config,
        )
        ack = (response.text or "").strip()
    except Exception as exc:
        # Network / API failure: fall back to a static acknowledgement so the
        # crisis path never silently dies.
        print(f"⚠️  Crisis ack LLM call failed ({exc}); using fallback.")
        ack = (
            "I hear you, and I'm really glad you reached out. What you're "
            "carrying sounds incredibly heavy. You are not alone in this, "
            "and your pain matters. Please don't go through this by yourself."
        )

    # --- Compose final reply -------------------------------------------------
    full_text = f"{ack}\n\n{CRISIS_RESOURCES_BLOCK}"

    return {
        "text":        full_text,
        "sources":     [],          # no RAG citations in crisis path
        "retrieved":   [],
        "prompt_used": None,
        "path":        "crisis",
        "classifier":  classifier_result,
    }


# ---------------------------------------------------------------------------
# Crisis logging — JSONL, append-only
# ---------------------------------------------------------------------------
def _log_crisis_event(
    user_message: str,
    classifier_result: dict,
    response_text: str,
) -> None:
    """Append one record to data/processed/crisis_log.jsonl for safety review."""
    record = {
        "timestamp_utc":     datetime.now(timezone.utc).isoformat(),
        "user_message":      user_message,
        "classifier_label":  classifier_result.get("label"),
        "classifier_conf":   classifier_result.get("confidence"),
        "classifier_method": classifier_result.get("method"),
        "response_excerpt":  response_text[:300],   # truncated, not the full thing
    }
    try:
        with open(CRISIS_LOG_PATH, "a", encoding="utf-8") as f:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
    except Exception as exc:
        # Logging must NEVER break the user-facing path.
        print(f"⚠️  Failed to log crisis event ({exc}); continuing anyway.")


# ---------------------------------------------------------------------------
# Orchestrator — single entry point
# ---------------------------------------------------------------------------
def respond(
    user_message: str,
    history: list[dict] = None,
    crisis_threshold: float = 0.5,
) -> dict:
    """
    Full pipeline. Cleans → classifies → routes → returns unified dict.

    Parameters
    ----------
    user_message : str
        Raw user input.
    history : list of {'role': 'user'|'assistant', 'content': str}, optional
        Prior conversation. The crisis branch ignores history; the RAG
        branch uses it.
    crisis_threshold : float
        Probability cutoff for the crisis branch. 0.5 is more sensitive
        than the 0.7 used in the original predict.py — for a mental health
        context, missing a real crisis is worse than over-flagging.

    Returns
    -------
    dict with keys:
        text, sources, retrieved, prompt_used, path, classifier
    where path is either 'crisis' or 'rag'.
    """
    if history is None:
        history = []

    # --- 1. Clean ------------------------------------------------------------
    cleaned = clean_text(user_message)
    if not cleaned:
        # Pathological input (empty after cleaning) — short-circuit gently.
        return {
            "text":        "Could you tell me a bit more about what's on your mind?",
            "sources":     [],
            "retrieved":   [],
            "prompt_used": None,
            "path":        "fallback",
            "classifier":  None,
        }

    # --- 2. Classify ---------------------------------------------------------
    classifier_result = is_crisis(cleaned, threshold=crisis_threshold)

    # --- 3. Route ------------------------------------------------------------
    if classifier_result["is_crisis"]:
        result = build_crisis_response(cleaned, classifier_result)
        _log_crisis_event(cleaned, classifier_result, result["text"])
        return result

    # RAG branch
    result = generate_response(cleaned, history=history)
    result["path"] = "rag"
    result["classifier"] = classifier_result
    return result


# ---------------------------------------------------------------------------
# Smoke test — mix of normal and crisis messages
# ---------------------------------------------------------------------------
test_cases = [
    # Normal — should go down RAG path
    ("I've been really stressed about exams and can't focus", "rag"),
    ("I just feel kind of empty lately", "rag"),
    # Crisis — should go down crisis path
    ("I don't want to be here anymore. I can't keep doing this.", "crisis"),
    ("I've been thinking about ending it all", "crisis"),
]

for i, (msg, expected_path) in enumerate(test_cases, 1):
    print("\n" + "=" * 80)
    print(f"TEST {i}: ({expected_path.upper()} expected)  {msg}")
    print("=" * 80)

    result = respond(msg, history=[])

    path_icon = "🚨" if result["path"] == "crisis" else "💬"
    match = "✅" if result["path"] == expected_path else "❌"
    print(f"\n{match} routed to: {path_icon} {result['path']}")

    if result.get("classifier"):
        c = result["classifier"]
        print(f"   classifier: label={c['label']}  "
              f"conf={c['confidence']:.3f}  method={c['method']}")

    print(f"\n💬 REPLY:\n{result['text']}")

    if result["sources"]:
        print(f"\n📎 CITED ({len(result['sources'])}):")
        for s in result["sources"]:
            print(f"   - {s['source']:14s} | {s['topic_or_category']:25s} | "
                  f"{s['text'][:80]}")

print("\n" + "=" * 80)
print(f"Crisis log location: {CRISIS_LOG_PATH}")
if CRISIS_LOG_PATH.exists():
    n_entries = sum(1 for _ in open(CRISIS_LOG_PATH, encoding="utf-8"))
    print(f"Crisis log now contains {n_entries} entries")

NOTE- some additional changes has been added to the rag.py file to make it app friendly

# Perform EDA

"""
eda.py
------
Exploratory data analysis figures for the report.

Generates 7 PNG figures into reports/figures/:
    fig1_class_distribution.png   — crisis vs non-crisis bar chart
    fig2_text_length.png          — message length distributions overlaid
    fig3_wordcloud_crisis.png     — most common words in crisis posts
    fig4_wordcloud_non_crisis.png — most common words in non-crisis posts
    fig5_counsel_topics.png       — top 15 Counsel Chat topics
    fig6_answer_length.png        — therapist answer length distribution
    fig7_knowledge_base.png       — KB entries grouped by category

Run from anywhere:
    python eda.py
    poetry run python assignment3/eda.py
"""

import json

import matplotlib.pyplot as plt
import pandas as pd
from wordcloud import STOPWORDS, WordCloud

from config import EXTERNAL_DATA_DIR, FIGURES_DIR, PROCESSED_DATA_DIR


# ---------------------------------------------------------------------------
# Paths (resolved from config.py — independent of cwd)
# ---------------------------------------------------------------------------
CRISIS_TRAIN_PATH = PROCESSED_DATA_DIR / "crisis_train.csv"
COUNSEL_PATH      = PROCESSED_DATA_DIR / "counsel_chat_clean.csv"
KB_PATH           = EXTERNAL_DATA_DIR / "knowledge_base.json"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------------------------
# Load crisis training data
# ---------------------------------------------------------------------------
df = pd.read_csv(CRISIS_TRAIN_PATH)


# ---------------------------------------------------------------------------
# FIGURE 1: Class Distribution
# ---------------------------------------------------------------------------
counts = df['label'].value_counts()

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(
    ['Non-Crisis', 'Crisis'],
    [counts['non-suicide'], counts['suicide']],
    color=['#10b981', '#ef4444'],
    edgecolor='white',
    width=0.5,
)
for bar, count in zip(bars, [counts['non-suicide'], counts['suicide']]):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 500,
        f'{count:,}',
        ha='center', fontsize=12, fontweight='bold',
    )
ax.set_title('Class Distribution: Crisis vs Non-Crisis Posts',
             fontsize=14, fontweight='bold', pad=15)
ax.set_ylabel('Number of Posts', fontsize=12)
ax.set_xlabel('Class', fontsize=12)
ax.set_ylim(0, 95000)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig1_class_distribution.png', dpi=150)
plt.close()


# ---------------------------------------------------------------------------
# FIGURE 2: Text Length Distribution
# ---------------------------------------------------------------------------
df['text_length'] = df['text'].astype(str).apply(len)
crisis     = df[df['label'] == 'suicide']['text_length']
non_crisis = df[df['label'] == 'non-suicide']['text_length']

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(non_crisis.clip(upper=2000), bins=60,
        alpha=0.6, color='#10b981', label='Non-Crisis', edgecolor='none')
ax.hist(crisis.clip(upper=2000), bins=60,
        alpha=0.6, color='#ef4444', label='Crisis', edgecolor='none')
ax.axvline(crisis.median(), color='#ef4444', linestyle='--',
           linewidth=2, label=f'Crisis median: {int(crisis.median())}')
ax.axvline(non_crisis.median(), color='#10b981', linestyle='--',
           linewidth=2, label=f'Non-Crisis median: {int(non_crisis.median())}')
ax.set_title('Text Length Distribution: Crisis vs Non-Crisis',
             fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Message Length (characters)', fontsize=12)
ax.set_ylabel('Number of Posts', fontsize=12)
ax.legend(fontsize=11)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig2_text_length.png', dpi=150)
plt.close()


# ---------------------------------------------------------------------------
# FIGURE 3: Word Cloud — Crisis Posts
# ---------------------------------------------------------------------------
stopwords = set(STOPWORDS)
stopwords.update([
    'will', 'one', 'now', 'just', 'like',
    'know', 'really', 'want', 'feel', 'get',
    'go', 'im', 'ive', 'dont', 'cant',
])

crisis_text = ' '.join(df[df['label'] == 'suicide']['text'].astype(str).tolist())
wc = WordCloud(
    width=1000, height=500,
    background_color='black',
    colormap='Reds',
    stopwords=stopwords,
    max_words=100,
    collocations=False,
).generate(crisis_text)

fig, ax = plt.subplots(figsize=(12, 6))
ax.imshow(wc, interpolation='bilinear')
ax.axis('off')
ax.set_title('Most Common Words: Crisis Posts',
             fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig3_wordcloud_crisis.png', dpi=150)
plt.close()


# ---------------------------------------------------------------------------
# FIGURE 4: Word Cloud — Non-Crisis Posts
# ---------------------------------------------------------------------------
non_crisis_text = ' '.join(
    df[df['label'] == 'non-suicide']['text'].astype(str).tolist()
)
wc2 = WordCloud(
    width=1000, height=500,
    background_color='black',
    colormap='Greens',
    stopwords=stopwords,
    max_words=100,
    collocations=False,
).generate(non_crisis_text)

fig, ax = plt.subplots(figsize=(12, 6))
ax.imshow(wc2, interpolation='bilinear')
ax.axis('off')
ax.set_title('Most Common Words: Non-Crisis Posts',
             fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig4_wordcloud_non_crisis.png', dpi=150)
plt.close()


# ---------------------------------------------------------------------------
# FIGURE 5: Counsel Chat Topic Distribution
# ---------------------------------------------------------------------------
df2 = pd.read_csv(COUNSEL_PATH)
print(df2.columns.tolist())
print(df2.head(2))

topic_counts = df2['topic'].value_counts().head(15)
fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(
    topic_counts.index[::-1],
    topic_counts.values[::-1],
    color='#10b981',
    edgecolor='white',
)
for bar, val in zip(bars, topic_counts.values[::-1]):
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height() / 2,
            str(val), va='center', fontsize=9)
ax.set_title('Top 15 Topics: Counsel Chat Dataset',
             fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Number of Questions', fontsize=12)
ax.set_ylabel('Topic', fontsize=12)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig5_counsel_topics.png', dpi=150)
plt.close()


# ---------------------------------------------------------------------------
# FIGURE 6: Therapist Answer Length
# ---------------------------------------------------------------------------
df2['answer_length'] = df2['answerText'].astype(str).apply(len)

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(df2['answer_length'].clip(upper=3000), bins=50,
        color='#10b981', edgecolor='white', alpha=0.85)
ax.axvline(df2['answer_length'].median(), color='#ef4444',
           linestyle='--', linewidth=2,
           label=f"Median: {int(df2['answer_length'].median())} chars")
ax.set_title('Therapist Answer Length: Counsel Chat Dataset',
             fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Answer Length (characters)', fontsize=12)
ax.set_ylabel('Number of Answers', fontsize=12)
ax.legend(fontsize=11)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig6_answer_length.png', dpi=150)
plt.close()


# ---------------------------------------------------------------------------
# FIGURE 7: Knowledge Base Category Breakdown
# ---------------------------------------------------------------------------
with open(KB_PATH, "r", encoding="utf-8") as f:
    kb = json.load(f)

categories = {}
for entry in kb:
    cat = entry.get("category", "Unknown")
    categories[cat] = categories.get(cat, 0) + 1

cat_df = pd.Series(categories).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 6))
bars = ax.barh(cat_df.index, cat_df.values,
               color='#6366f1', edgecolor='white', alpha=0.9)
for bar, val in zip(bars, cat_df.values):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height() / 2,
            str(val), va='center', fontsize=10, fontweight='bold')
ax.set_title('Knowledge Base: Entries by Category',
             fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Number of Entries', fontsize=12)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig7_knowledge_base.png', dpi=150)
plt.close()

print(f"\n✅ Generated 7 figures in {FIGURES_DIR}")